In [1]:
# Clone the repository safely

%cd /content

!rm -rf XRL-for-optimising-battery-schedules
!git clone -b master https://github.com/farhanshaikh199704/XRL-for-optimising-battery-schedules.git

%cd /content/XRL-for-optimising-battery-schedules

import os

print("Repository:", os.getcwd())

print("Branch:")
!git branch --show-current

print("Latest commit:")
!git log -1 --oneline

/content
Cloning into 'XRL-for-optimising-battery-schedules'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 195 (delta 3), reused 3 (delta 3), pack-reused 189 (from 2)
Receiving objects: 100% (195/195), 84.66 MiB | 30.42 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/XRL-for-optimising-battery-schedules
Repository: /content/XRL-for-optimising-battery-schedules
Branch:
master
Latest commit:
83fe427 (HEAD -> master, origin/master, origin/HEAD) Included trained RL model files (dqn, dqn_tft, ppo)


In [2]:
# ============================================================
# Load repo + H2 forecast cache
# ============================================================

import sys
import numpy as np
import pandas as pd
import torch

REPO_ROOT = "/content/XRL-for-optimising-battery-schedules"

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# H2 TFT forecast file you just generated
H2_FORECAST_PATH = "/content/XRL-for-optimising-battery-schedules/forecasters/trained_models/tft_36h/tft_h2_2025_test_predictions (1).csv"

# Existing trained RL models
DQN_MODEL_PATH = os.path.join(
    REPO_ROOT,
    "log", "causal_rl", "dqn_tft", "run_01", "model_final.zip"
)

PPO_MODEL_PATH = os.path.join(
    REPO_ROOT,
    "log", "causal_rl", "ppo_tft", "run_01", "model_final.zip"
)

SAC_MODEL_PATH = os.path.join(
    REPO_ROOT,
    "log", "causal_rl", "sac_tft", "run_01", "model_final.zip"
)

# Load H2 forecasts
h2_forecast = pd.read_csv(H2_FORECAST_PATH)

print("=" * 70)
print("DEMO3_vF — H2 FINAL RL EVALUATION")
print("=" * 70)

print(f"H2 forecast rows : {len(h2_forecast):,}")
print(f"Start             : {h2_forecast['timestamp'].iloc[0]}")
print(f"End               : {h2_forecast['timestamp'].iloc[-1]}")

print()
print("DQN model exists :", os.path.exists(DQN_MODEL_PATH))
print("PPO model exists :", os.path.exists(PPO_MODEL_PATH))
print("SAC model exists :", os.path.exists(SAC_MODEL_PATH))

print()
print("H2 forecast columns:")
print(h2_forecast.columns.tolist())

DEMO3_vF — H2 FINAL RL EVALUATION
H2 forecast rows : 4,416
Start             : 01-07-2025 00:00
End               : 31-12-2025 23:00

DQN model exists : True
PPO model exists : True
SAC model exists : True

H2 forecast columns:
['delivery_date', 'timestamp', 'time_idx', 'p10', 'p50', 'p90', 'uncertainty', 'actual_pool_price']


In [3]:
# ============================================================
# Build H2 RL dataset with exact training TFT structure
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

REPO_ROOT = "/content/XRL-for-optimising-battery-schedules"

BASE_CSV = os.path.join(
    REPO_ROOT,
    "data",
    "German data",
    "german_epex_dam_idm_2022_2025.csv"
)

TRAIN_TFT_CSV = os.path.join(
    REPO_ROOT,
    "data",
    "rl",
    "german_rl_combined_tft_2022_2025H1.csv"
)

H2_FORECAST_PATH = os.path.join(
    REPO_ROOT,
    "forecasters",
    "trained_models",
    "tft_36h",
    "tft_h2_2025_test_predictions (1).csv"
)

H2_RL_CSV = os.path.join(
    REPO_ROOT,
    "data",
    "rl",
    "german_rl_h2_2025.csv"
)

# ------------------------------------------------------------
# Load training dataset
# Used only to obtain the exact column names and ordering
# ------------------------------------------------------------

train_tft = pd.read_csv(TRAIN_TFT_CSV)

# ------------------------------------------------------------
# Load H2 TFT forecasts
# ------------------------------------------------------------

h2_forecast = pd.read_csv(H2_FORECAST_PATH)

print("=" * 70)
print("BUILDING H2 RL DATASET")
print("=" * 70)
print(f"Training dataset shape : {train_tft.shape}")
print(f"H2 forecast shape      : {h2_forecast.shape}")
print()

# ------------------------------------------------------------
# Exact TFT column names used by the training dataset
# ------------------------------------------------------------

TFT_COLS_EXACT = (
    [f"TFT_p10_h{i:02d}" for i in range(24)]
    + [f"TFT_p50_h{i:02d}" for i in range(24)]
    + [f"TFT_p90_h{i:02d}" for i in range(24)]
    + [f"TFT_uncertainty_h{i:02d}" for i in range(24)]
)

# Confirm these columns actually exist in training data
missing_training_tft = [
    c for c in TFT_COLS_EXACT
    if c not in train_tft.columns
]

if missing_training_tft:
    raise ValueError(
        "Expected TFT columns are missing from the training dataset:\n"
        + "\n".join(missing_training_tft)
    )

# ------------------------------------------------------------
# Prepare H2 forecast data
# ------------------------------------------------------------

fc = h2_forecast.copy()

# Forecast timestamp is DD-MM-YYYY HH:MM
fc["timestamp"] = pd.to_datetime(
    fc["timestamp"],
    format="%d-%m-%Y %H:%M"
)

# Forecast delivery_date is DD-MM-YYYY
fc["delivery_date"] = pd.to_datetime(
    fc["delivery_date"],
    dayfirst=True
).dt.normalize()

# Sort chronologically
fc = fc.sort_values(
    ["delivery_date", "timestamp"]
).copy()

# ------------------------------------------------------------
# Check forecast columns
# ------------------------------------------------------------

required_forecast_cols = [
    "delivery_date",
    "timestamp",
    "p10",
    "p50",
    "p90",
    "uncertainty",
]

missing_forecast_cols = [
    c for c in required_forecast_cols
    if c not in fc.columns
]

if missing_forecast_cols:
    raise ValueError(
        f"Missing H2 forecast columns: {missing_forecast_cols}"
    )

# ------------------------------------------------------------
# Confirm exactly 24 forecast hours per delivery day
# ------------------------------------------------------------

hours_per_day = fc.groupby("delivery_date").size()

if not (hours_per_day == 24).all():
    bad_days = hours_per_day[hours_per_day != 24]

    raise ValueError(
        "Some delivery days do not have exactly 24 forecast hours:\n"
        f"{bad_days}"
    )

print(
    f"H2 delivery days      : {fc['delivery_date'].nunique()}"
)
print(
    f"Forecast rows          : {len(fc):,}"
)
print()

# ------------------------------------------------------------
# Build one 24-hour TFT forecast vector per delivery day
#
# Each delivery day gets:
#   24 × p10
#   24 × p50
#   24 × p90
#   24 × uncertainty
#
# This gives the exact 96-dimensional TFT state extension.
# ------------------------------------------------------------

daily_vectors = []

for delivery_date, day in fc.groupby(
    "delivery_date",
    sort=True
):

    day = day.sort_values("timestamp")

    row = {
        "delivery_date": delivery_date
    }

    for h in range(24):

        row[f"TFT_p10_h{h:02d}"] = float(
            day.iloc[h]["p10"]
        )

        row[f"TFT_p50_h{h:02d}"] = float(
            day.iloc[h]["p50"]
        )

        row[f"TFT_p90_h{h:02d}"] = float(
            day.iloc[h]["p90"]
        )

        row[f"TFT_uncertainty_h{h:02d}"] = float(
            day.iloc[h]["uncertainty"]
        )

    daily_vectors.append(row)

daily_tft = pd.DataFrame(daily_vectors)

# ------------------------------------------------------------
# Load base German market dataset
# ------------------------------------------------------------

base = pd.read_csv(BASE_CSV)

base["timestamp"] = pd.to_datetime(
    base["timestamp"],
    format="%d-%m-%Y %H:%M"
)

# Delivery date corresponding to each hourly observation
base["delivery_date"] = (
    base["timestamp"].dt.normalize()
)

# ------------------------------------------------------------
# Select H2 test period only
#
# IMPORTANT:
# H2 remains completely separate from model development.
# ------------------------------------------------------------

h2_base = base[
    (base["timestamp"] >= "2025-07-01 00:00:00") &
    (base["timestamp"] <= "2025-12-31 23:00:00")
].copy()

print(
    f"Base H2 rows          : {len(h2_base):,}"
)

# ------------------------------------------------------------
# Merge the daily 96-dimensional TFT vector onto each
# hourly observation belonging to that delivery day
# ------------------------------------------------------------

h2_rl = h2_base.merge(
    daily_tft,
    on="delivery_date",
    how="inner"
)

# ------------------------------------------------------------
# Use EXACTLY the same column order as training
# ------------------------------------------------------------

training_columns = train_tft.columns.tolist()

missing_columns = [
    c for c in training_columns
    if c not in h2_rl.columns
]

if missing_columns:
    raise ValueError(
        "Missing columns required by the training dataset:\n"
        + "\n".join(missing_columns)
    )

h2_rl = h2_rl[training_columns].copy()

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

if len(h2_rl) != 4416:
    raise ValueError(
        f"Expected 4416 H2 rows, got {len(h2_rl)}"
    )

if len(h2_rl.columns) != 126:
    raise ValueError(
        f"Expected 126 columns, got {len(h2_rl.columns)}"
    )

if not h2_rl["timestamp"].is_monotonic_increasing:
    raise ValueError(
        "H2 timestamps are not sorted."
    )

if not h2_rl["delivery_date"].is_monotonic_increasing:
    raise ValueError(
        "H2 delivery dates are not sorted."
    )

# Check TFT values
if not np.isfinite(
    h2_rl[TFT_COLS_EXACT].to_numpy(
        dtype=np.float32
    )
).all():

    raise ValueError(
        "NaN or Inf detected in TFT features."
    )

# Check 24 hourly observations per delivery day
rows_per_day = h2_rl.groupby(
    "delivery_date"
).size()

if not (rows_per_day == 24).all():

    bad_days = rows_per_day[
        rows_per_day != 24
    ]

    raise ValueError(
        "Some H2 delivery days do not contain "
        f"exactly 24 observations:\n{bad_days}"
    )

# ------------------------------------------------------------
# Confirm exact structural match with training dataset
# ------------------------------------------------------------

structure_match = (
    list(train_tft.columns)
    == list(h2_rl.columns)
)

if not structure_match:
    raise ValueError(
        "H2 dataset column structure does not match "
        "the training dataset."
    )

# ------------------------------------------------------------
# H2 RL dataset
# ------------------------------------------------------------

h2_rl.to_csv(
    H2_RL_CSV,
    index=False
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print()
print("=" * 70)
print("H2 RL DATASET CREATED")
print("=" * 70)

print(f"Rows             : {len(h2_rl):,}")
print(f"Columns          : {len(h2_rl.columns)}")
print(f"Market variables : 20")
print(f"TFT variables    : {len(TFT_COLS_EXACT)}")
print(
    f"Delivery days    : {h2_rl['delivery_date'].nunique()}"
)
print(
    f"Start            : {h2_rl['timestamp'].iloc[0]}"
)
print(
    f"End              : {h2_rl['timestamp'].iloc[-1]}"
)
print(
    f"Saved to         : {H2_RL_CSV}"
)
print()
print(
    f"TFT structure matches training: {structure_match}"
)

BUILDING H2 RL DATASET
Training dataset shape : (26304, 126)
H2 forecast shape      : (4416, 8)

H2 delivery days      : 184
Forecast rows          : 4,416

Base H2 rows          : 4,416

H2 RL DATASET CREATED
Rows             : 4,416
Columns          : 126
Market variables : 20
TFT variables    : 96
Delivery days    : 184
Start            : 2025-07-01 00:00:00
End              : 2025-12-31 23:00:00
Saved to         : /content/XRL-for-optimising-battery-schedules/data/rl/german_rl_h2_2025.csv

TFT structure matches training: True


In [4]:
import sys
import importlib.util
import subprocess

print("Python:", sys.version)
print()

packages = [
    "numpy",
    "pandas",
    "scipy",
    "torch",
    "gymnasium",
    "stable_baselines3",
    "pytorch_forecasting",
    "lightning",
    "shap",
    "matplotlib",
    "seaborn",
    "optuna",
]

for package in packages:
    spec = importlib.util.find_spec(package)

    if spec is None:
        print(f"{package:25s} NOT INSTALLED")
    else:
        try:
            module = __import__(package)
            version = getattr(module, "__version__", "version unknown")
            print(f"{package:25s} {version}")
        except Exception as e:
            print(f"{package:25s} IMPORT ERROR: {type(e).__name__}: {e}")

print()
print("===== PyTorch / GPU =====")

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

numpy                     2.1.3
pandas                    2.2.3
scipy                     1.16.3
torch                     2.11.0+cu130
gymnasium                 1.3.0
stable_baselines3         NOT INSTALLED
pytorch_forecasting       NOT INSTALLED
lightning                 NOT INSTALLED
shap                      0.52.0
matplotlib                3.10.0
seaborn                   0.13.2
optuna                    NOT INSTALLED

===== PyTorch / GPU =====
PyTorch: 2.11.0+cu130
CUDA available: True
CUDA: 13.0
GPU: NVIDIA A100-SXM4-40GB
GPU memory: 39.49 GB


In [5]:
!pip install -q "stable-baselines3"

import stable_baselines3 as sb3
import gymnasium

print("Stable-Baselines3:", sb3.__version__)
print("Gymnasium:", gymnasium.__version__)

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 5.5 MB/s eta 0:00:00
Stable-Baselines3: 2.9.0
Gymnasium: 1.3.0
PyTorch: 2.11.0+cu130
CUDA: 13.0


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# ============================================================
# Load frozen DQN / PPO / SAC models
# ============================================================

from stable_baselines3 import DQN, PPO, SAC

# ------------------------------------------------------------
# Model paths
# ------------------------------------------------------------

DQN_MODEL_PATH = os.path.join(
    REPO_ROOT,
    "log",
    "causal_rl",
    "dqn_tft",
    "run_01",
    "model_final.zip"
)

PPO_MODEL_PATH = os.path.join(
    REPO_ROOT,
    "log",
    "causal_rl",
    "ppo_tft",
    "run_01",
    "model_final.zip"
)

SAC_MODEL_PATH = os.path.join(
    REPO_ROOT,
    "log",
    "causal_rl",
    "sac_tft",
    "run_01",
    "model_final.zip"
)

# ------------------------------------------------------------
# Load frozen models
# ------------------------------------------------------------

print("=" * 70)
print("LOADING FROZEN RL MODELS")
print("=" * 70)

dqn_model = DQN.load(
    DQN_MODEL_PATH,
    device="cuda"
)

ppo_model = PPO.load(
    PPO_MODEL_PATH,
    device="cuda"
)

sac_model = SAC.load(
    SAC_MODEL_PATH,
    device="cuda"
)

print("DQN loaded : OK")
print("PPO loaded : OK")
print("SAC loaded : OK")

print()
print("Model observation spaces:")
print("DQN:", dqn_model.observation_space)
print("PPO:", ppo_model.observation_space)
print("SAC:", sac_model.observation_space)

print()
print("Model action spaces:")
print("DQN:", dqn_model.action_space)
print("PPO:", ppo_model.action_space)
print("SAC:", sac_model.action_space)

LOADING FROZEN RL MODELS
DQN loaded : OK
PPO loaded : OK
SAC loaded : OK

Model observation spaces:
DQN: Box([ 0.0000000e+00 -5.0000000e+02 -5.0000000e+02 -5.0000000e+02
 -5.0000000e+02 -5.0000000e+02 -1.5217999e+02 -2.5013050e+01
 -2.5013050e+01 -5.0000000e+02  0.0000000e+00  4.6500000e+01
  1.2500000e+00  3.0902750e+04 -8.3237500e+03 -1.0000000e+00
 -1.0000000e+00 -9.7492802e-01 -9.0096903e-01 -1.0000000e+00
 -1.0000000e+00 -2.7325833e+02 -2.9348975e+02 -2.9221310e+02
 -2.8028793e+02 -2.6310123e+02 -2.4421890e+02 -2.3110083e+02
 -2.2053491e+02 -2.2208847e+02 -2.4424777e+02 -2.7345227e+02
 -3.0256390e+02 -3.2926474e+02 -3.4639139e+02 -3.4339053e+02
 -3.1457224e+02 -2.5616043e+02 -1.6812009e+02 -9.8092926e+01
 -6.4688126e+01 -6.5630859e+01 -9.9725464e+01 -1.6987881e+02
 -2.5370926e+02 -9.9321426e+01 -1.2561038e+02 -1.2943713e+02
 -1.1984277e+02 -1.0208072e+02 -7.7838303e+01 -6.8385468e+01
 -7.2946075e+01 -8.0487183e+01 -8.7242981e+01 -9.4375275e+01
 -1.2277341e+02 -1.4622604e+02 -1.582

/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ============================================================
# Check training normalization artifacts
# ============================================================

import os
import inspect


RUN_DIRS={
    "DQN": os.path.dirname(DQN_MODEL_PATH),
    "PPO": os.path.dirname(PPO_MODEL_PATH),
    "SAC": os.path.dirname(SAC_MODEL_PATH),
}

print("=" * 70)
print("TRAINING RUN ARTIFACT CHECK")
print("=" * 70)

for name, run_dir in RUN_DIRS.items():
    print(f"\n{name}")
    print("-" * 50)
    print(f"Run directory: {run_dir}")


from utils.make_env import make_env

print("=" * 70)
print("REPOSITORY make_env() CONFIGURATION")
print("=" * 70)

print(inspect.getsource(make_env))

TRAINING RUN ARTIFACT CHECK

DQN
--------------------------------------------------
Run directory: /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_tft/run_01

PPO
--------------------------------------------------
Run directory: /content/XRL-for-optimising-battery-schedules/log/causal_rl/ppo_tft/run_01

SAC
--------------------------------------------------
Run directory: /content/XRL-for-optimising-battery-schedules/log/causal_rl/sac_tft/run_01
REPOSITORY make_env() CONFIGURATION
def make_env(env,
             env_kwargs: dict,
             tracking: bool = False,
             allow_early_resets: bool = True,
             path: Optional[str] = None,
             perfect_forecasts: Optional[list] = None,
             forecasts: Optional[Dict[str, Any]] = None,
             flatten_obs: bool = True,
             discrete_actions: Optional[list] = None,
             norm_obs: bool = True,
             norm_reward: bool = True,
             gamma: float = 0.99,
           

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ============================================================
# CELL 6 — CORRECTED TRAINING-ONLY NORMALIZATION RECONSTRUCTION
# ============================================================

import os
import sys
import numpy as np
import pandas as pd

from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from gymnasium.wrappers import FlattenObservation
from gymnasium import spaces

REPO = "/content/XRL-for-optimising-battery-schedules"

FULL_DATA = os.path.join(
    REPO, "data", "rl", "german_rl_combined_tft_2022_2025H1.csv"
)

TRAIN_ONLY_DATA = os.path.join(
    REPO, "data", "rl", "_normalization_train_only.csv"
)

# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_tft

# ------------------------------------------------------------
# Exact TFT feature order
# ------------------------------------------------------------
TFT_COLS = (
    [f"TFT_p10_h{i:02d}" for i in range(24)]
    + [f"TFT_p50_h{i:02d}" for i in range(24)]
    + [f"TFT_p90_h{i:02d}" for i in range(24)]
    + [f"TFT_uncertainty_h{i:02d}" for i in range(24)]
)

# ------------------------------------------------------------
# Create TRAIN-ONLY dataset
# ------------------------------------------------------------
df = pd.read_csv(FULL_DATA)
df["timestamp"] = pd.to_datetime(df["timestamp"])

train_mask = (
    (df["timestamp"] >= "2022-07-01 00:00:00")
    & (df["timestamp"] <= "2024-12-31 23:00:00")
)

train_df = df.loc[train_mask].copy()

assert len(train_df) == 21960, \
    f"Expected 21,960 rows, got {len(train_df)}"

train_df.to_csv(TRAIN_ONLY_DATA, index=False)

print("=" * 70)
print("TRAINING-ONLY DATASET")
print("=" * 70)
print(f"Rows  : {len(train_df):,}")
print(f"Start : {train_df['timestamp'].min()}")
print(f"End   : {train_df['timestamp'].max()}")
print(f"Saved : {TRAIN_ONLY_DATA}")

# ------------------------------------------------------------
# Causal TFT environment
# ------------------------------------------------------------
class CausalTFTBatteryEnv(FreeBatteryEnv):

    def __init__(self, *args, **kwargs):
        kwargs["tft_24h"] = None
        super().__init__(*args, **kwargs)

        tft_df = pd.read_csv(self.data_file)

        missing = [c for c in TFT_COLS if c not in tft_df.columns]
        if missing:
            raise ValueError(f"Missing TFT columns: {missing}")

        self.tft_data = tft_df[TFT_COLS].to_numpy(dtype=np.float32)

        if len(self.tft_data) != len(self.data):
            raise ValueError(
                f"TFT rows ({len(self.tft_data)}) "
                f"!= environment rows ({len(self.data)})"
            )

        if not np.isfinite(self.tft_data).all():
            raise ValueError("Non-finite TFT values found")

        self.observation_space["tft_forecasts"] = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(96,),
            dtype=np.float32,
        )

    def _get_obs(self):
        obs, done = super()._get_obs()
        idx = min(self.count, len(self.tft_data) - 1)
        obs["tft_forecasts"] = self.tft_data[idx]
        return obs, done


# ------------------------------------------------------------
# Environment configuration
# ------------------------------------------------------------
env_kwargs = dict(de_bat_tft)
env_kwargs["data_file"] = TRAIN_ONLY_DATA

print("\nEnvironment:")
print(f"  Data file : {TRAIN_ONLY_DATA}")
print("  Expected observation dimension : 117")

# ------------------------------------------------------------
# Build same wrapper chain used during training:
#
# Monitor
#   -> FlattenObservation
#   -> DummyVecEnv
#   -> VecNormalize
# ------------------------------------------------------------
def make_reconstruction_env():
    env = CausalTFTBatteryEnv(
        **env_kwargs,
        tracking=False
    )
    env = Monitor(env)
    env = FlattenObservation(env)
    return env


dummy = DummyVecEnv([make_reconstruction_env])

vecnorm = VecNormalize(
    dummy,
    norm_obs=True,
    norm_reward=True,
    gamma=0.99,
)

# ------------------------------------------------------------
# Replay EXACTLY the 21,960 training observations
# ------------------------------------------------------------
obs = vecnorm.reset()

steps = 0
raw_reward_sum = 0.0

while True:

    # Zero action is sufficient for collecting observation
    # normalization statistics.
    action = np.array([[0.0]], dtype=np.float32)

    obs, reward, done, info = vecnorm.step(action)

    raw_reward_sum += float(reward[0])
    steps += 1

    if done[0]:
        break

print("\nReplay complete:")
print(f"  Steps replayed : {steps:,}")
print(f"  Reward         : {raw_reward_sum:,.2f}")

assert steps == 21960, \
    f"Expected 21,960 steps, got {steps}"

# ------------------------------------------------------------
# Freeze statistics
# ------------------------------------------------------------
vecnorm.training = False
vecnorm.norm_reward = False

NORM_PATH = os.path.join(
    REPO,
    "data",
    "rl",
    "reconstructed_vecnormalize.pkl"
)

vecnorm.save(NORM_PATH)

print("\n" + "=" * 70)
print("NORMALIZATION RECONSTRUCTION COMPLETE")
print("=" * 70)
print(f"Saved          : {NORM_PATH}")
print(f"Obs mean shape : {vecnorm.obs_rms.mean.shape}")
print(f"Obs var shape  : {vecnorm.obs_rms.var.shape}")
print(f"Obs count      : {vecnorm.obs_rms.count:,.1f}")
print("=" * 70)

TRAINING-ONLY DATASET
Rows  : 21,960
Start : 2022-07-01 00:00:00
End   : 2024-12-31 23:00:00
Saved : /content/XRL-for-optimising-battery-schedules/data/rl/_normalization_train_only.csv

Environment:
  Data file : /content/XRL-for-optimising-battery-schedules/data/rl/_normalization_train_only.csv
  Expected observation dimension : 117


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Replay complete:
  Steps replayed : 21,960
  Reward         : 0.00

NORMALIZATION RECONSTRUCTION COMPLETE
Saved          : /content/XRL-for-optimising-battery-schedules/data/rl/reconstructed_vecnormalize.pkl
Obs mean shape : (117,)
Obs var shape  : (117,)
Obs count      : 21,961.0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ============================================================
# DQN + CAUSAL TFT — H2 2025 EVALUATION
# ============================================================

import os
import sys
import numpy as np
import pandas as pd
import gymnasium as gym

from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import (
    DummyVecEnv,
    VecNormalize,
)
from gymnasium.wrappers import FlattenObservation

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
REPO = "/content/XRL-for-optimising-battery-schedules"

TRAIN_DATA = os.path.join(
    REPO,
    "data/rl/german_rl_train_2022H2_2024.csv"
)

H2_DATA = os.path.join(
    REPO,
    "data/rl/german_rl_h2_2025.csv"
)

NORM_PATH = os.path.join(
    REPO,
    "data/rl/reconstructed_vecnormalize.pkl"
)

MODEL_PATH = os.path.join(
    REPO,
    "log/causal_rl/dqn_tft/run_01/model_final.zip"
)

sys.path.insert(0, REPO)

from envs.environments import FreeBatteryEnv


# ------------------------------------------------------------
# TFT state columns
# ------------------------------------------------------------
TFT_COLS = (
    [f"TFT_p10_h{i:02d}" for i in range(24)]
    + [f"TFT_p50_h{i:02d}" for i in range(24)]
    + [f"TFT_p90_h{i:02d}" for i in range(24)]
    + [f"TFT_uncertainty_h{i:02d}" for i in range(24)]
)

assert len(TFT_COLS) == 96


# ------------------------------------------------------------
# Recreate the exact TFT-augmented environment
# ------------------------------------------------------------
class CausalTFTBatteryEnv(FreeBatteryEnv):

    def __init__(self, *args, **kwargs):

        # TFT is supplied directly through the RL data file.
        kwargs["tft_24h"] = None

        super().__init__(*args, **kwargs)

        tft_df = pd.read_csv(self.data_file)

        missing = [
            c for c in TFT_COLS
            if c not in tft_df.columns
        ]

        if missing:
            raise ValueError(
                f"Missing TFT columns: {missing}"
            )

        self.tft_data = (
            tft_df[TFT_COLS]
            .to_numpy(dtype=np.float32)
        )

        if len(self.tft_data) != len(self.data):
            raise ValueError(
                f"TFT rows ({len(self.tft_data)}) != "
                f"environment rows ({len(self.data)})"
            )

        if not np.isfinite(self.tft_data).all():
            raise ValueError(
                "Non-finite TFT values detected."
            )

        self.observation_space["tft_forecasts"] = gym.spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(96,),
            dtype=np.float32,
        )

    def _get_obs(self):

        obs, done = super()._get_obs()

        idx = min(
            self.count,
            len(self.tft_data) - 1
        )

        obs["tft_forecasts"] = self.tft_data[idx]

        return obs, done


# ------------------------------------------------------------
# Environment factory
# ------------------------------------------------------------
def make_raw_env(data_file):
    env_kwargs = dict(de_bat_tft)

    # Force the H2 data file
    env_kwargs["data_file"] = data_file

    # Use our causal 96-dim TFT observation
    env = CausalTFTBatteryEnv(
        **env_kwargs,
        tracking=True,
    )

    env = Monitor(env)
    env = FlattenObservation(env)
    return env


# ------------------------------------------------------------
# Build H2 vector environment
# ------------------------------------------------------------
h2_vec = DummyVecEnv([
    lambda: make_raw_env(H2_DATA)
])


# ------------------------------------------------------------
# Load reconstructed training normalization
# ------------------------------------------------------------
h2_vec = VecNormalize.load(
    NORM_PATH,
    h2_vec,
)

# IMPORTANT:
# Evaluation must not update normalization statistics.
h2_vec.training = False
h2_vec.norm_reward = False


# ------------------------------------------------------------
# Load frozen DQN
# ------------------------------------------------------------
model = DQN.load(
    MODEL_PATH,
    env=h2_vec,
    device="cuda",
)


# ------------------------------------------------------------
# Verify compatibility
# ------------------------------------------------------------
print("=" * 70)
print("DQN + CAUSAL TFT — H2 2025 EVALUATION")
print("=" * 70)

print("Observation dimension :", model.observation_space.shape[0])
print("Action space          :", model.action_space)
print("H2 rows               :", len(pd.read_csv(H2_DATA)))

assert model.observation_space.shape == (117,)
assert h2_vec.observation_space.shape == (117,)


# ------------------------------------------------------------
# Evaluate entire H2 period
# ------------------------------------------------------------
obs = h2_vec.reset()

episode_reward = 0.0
steps = 0
actions = []
rewards = []

while True:

    action, _ = model.predict(
        obs,
        deterministic=True,
    )

    obs, reward, done, info = h2_vec.step(action)

    reward_value = float(reward[0])

    episode_reward += reward_value
    steps += 1

    actions.append(action)
    rewards.append(reward_value)

    if done[0]:
        break


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
actions = np.asarray(actions).reshape(-1)
rewards = np.asarray(rewards)

print()
print("=" * 70)
print("H2 EVALUATION COMPLETE")
print("=" * 70)

print(f"Steps evaluated       : {steps:,}")
print(f"Cumulative reward     : {episode_reward:,.2f}")
print(f"Mean hourly reward    : {rewards.mean():,.4f}")
print(f"Reward std            : {rewards.std():,.4f}")

print()
print("Action distribution:")
print(
    pd.Series(actions)
    .value_counts()
    .sort_index()
)

print("=" * 70)

ValueError: Observation spaces do not match: Box([ 0.0000000e+00 -5.0000000e+02 -5.0000000e+02 -5.0000000e+02
 -5.0000000e+02 -5.0000000e+02 -1.5217999e+02 -2.5013050e+01
 -2.5013050e+01 -5.0000000e+02  0.0000000e+00  4.6500000e+01
  1.2500000e+00  3.0902750e+04 -8.3237500e+03 -1.0000000e+00
 -1.0000000e+00 -9.7492802e-01 -9.0096903e-01 -1.0000000e+00
 -1.0000000e+00 -2.7325833e+02 -2.9348975e+02 -2.9221310e+02
 -2.8028793e+02 -2.6310123e+02 -2.4421890e+02 -2.3110083e+02
 -2.2053491e+02 -2.2208847e+02 -2.4424777e+02 -2.7345227e+02
 -3.0256390e+02 -3.2926474e+02 -3.4639139e+02 -3.4339053e+02
 -3.1457224e+02 -2.5616043e+02 -1.6812009e+02 -9.8092926e+01
 -6.4688126e+01 -6.5630859e+01 -9.9725464e+01 -1.6987881e+02
 -2.5370926e+02 -9.9321426e+01 -1.2561038e+02 -1.2943713e+02
 -1.1984277e+02 -1.0208072e+02 -7.7838303e+01 -6.8385468e+01
 -7.2946075e+01 -8.0487183e+01 -8.7242981e+01 -9.4375275e+01
 -1.2277341e+02 -1.4622604e+02 -1.5829562e+02 -1.5416147e+02
 -1.3325870e+02 -8.7391937e+01 -2.01

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from stable_baselines3 import DQN

dqn_tmp = DQN.load(MODEL_PATH, device="cpu")

saved_space = dqn_tmp.observation_space
env_space = h2_vec.observation_space

print("Saved model low  :", saved_space.low)
print("Saved model high :", saved_space.high)

print("\nEnv low  :", env_space.low)
print("Env high :", env_space.high)

print("\nTFT saved bounds:")
print("low :", saved_space.low[21:25])
print("high:", saved_space.high[21:25])

print("\nTFT env bounds:")
print("low :", env_space.low[21:25])
print("high:", env_space.high[21:25])

Saved model low  : [ 0.0000000e+00 -5.0000000e+02 -5.0000000e+02 -5.0000000e+02
 -5.0000000e+02 -5.0000000e+02 -1.5217999e+02 -2.5013050e+01
 -2.5013050e+01 -5.0000000e+02  0.0000000e+00  4.6500000e+01
  1.2500000e+00  3.0902750e+04 -8.3237500e+03 -1.0000000e+00
 -1.0000000e+00 -9.7492802e-01 -9.0096903e-01 -1.0000000e+00
 -1.0000000e+00 -2.7325833e+02 -2.9348975e+02 -2.9221310e+02
 -2.8028793e+02 -2.6310123e+02 -2.4421890e+02 -2.3110083e+02
 -2.2053491e+02 -2.2208847e+02 -2.4424777e+02 -2.7345227e+02
 -3.0256390e+02 -3.2926474e+02 -3.4639139e+02 -3.4339053e+02
 -3.1457224e+02 -2.5616043e+02 -1.6812009e+02 -9.8092926e+01
 -6.4688126e+01 -6.5630859e+01 -9.9725464e+01 -1.6987881e+02
 -2.5370926e+02 -9.9321426e+01 -1.2561038e+02 -1.2943713e+02
 -1.1984277e+02 -1.0208072e+02 -7.7838303e+01 -6.8385468e+01
 -7.2946075e+01 -8.0487183e+01 -8.7242981e+01 -9.4375275e+01
 -1.2277341e+02 -1.4622604e+02 -1.5829562e+02 -1.5416147e+02
 -1.3325870e+02 -8.7391937e+01 -2.0125313e+01  1.8942669e+01
  1.0

In [ ]:
# Match the trained DQN's exact 117-D observation-space bounds.
# The first 21 dimensions already match; replace only the 96 TFT bounds.

base_env = h2_vec.envs[0].unwrapped

saved_low = dqn_tmp.observation_space.low.copy()
saved_high = dqn_tmp.observation_space.high.copy()

# Exact bounds learned/serialized with the trained DQN
base_env.observation_space = gym.spaces.Box(
    low=saved_low.astype(np.float32),
    high=saved_high.astype(np.float32),
    shape=(117,),
    dtype=np.float32,
)

# Rebuild the flattened/VecEnv wrapper so SB3 sees the corrected space
h2_vec = DummyVecEnv([
    lambda: make_raw_env(H2_DATA)
])

# Apply the exact saved bounds again after wrapper construction
h2_vec.envs[0].observation_space = gym.spaces.Box(
    low=saved_low.astype(np.float32),
    high=saved_high.astype(np.float32),
    shape=(117,),
    dtype=np.float32,
)

print("Corrected observation space:", h2_vec.observation_space)
print("Space matches saved DQN:",
      h2_vec.observation_space == dqn_tmp.observation_space)

Corrected observation space: Box([ 0.000000e+00 -6.108000e+01 -6.108000e+01 -6.108000e+01 -6.108000e+01
 -6.108000e+01 -4.000000e+01 -2.501305e+01 -2.501305e+01 -3.308250e+02
  6.700000e-01  3.704200e+02  4.750000e+00  3.323327e+04 -8.248330e+03
 -1.000000e+00 -1.000000e+00 -9.749280e-01 -9.009690e-01 -1.000000e+00
 -8.660250e-01          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf  

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Force the VecEnv's declared observation space to exactly match
# the observation space serialized inside the trained DQN.

h2_vec.observation_space = gym.spaces.Box(
    low=dqn_tmp.observation_space.low.copy(),
    high=dqn_tmp.observation_space.high.copy(),
    shape=dqn_tmp.observation_space.shape,
    dtype=np.float32,
)

print("Space matches saved DQN:",
      h2_vec.observation_space == dqn_tmp.observation_space)

print("H2 observation shape:", h2_vec.observation_space.shape)

Space matches saved DQN: True
H2 observation shape: (117,)


In [ ]:
# Match the trained DQN's 5-action interface exactly.

DISCRETE_ACTIONS = [
    np.array([-1.0]),
    np.array([-0.5]),
    np.array([0.0]),
    np.array([0.5]),
    np.array([1.0]),
]

# Add the same action wrapper used during DQN training
h2_vec.envs[0] = DiscreteActions(
    h2_vec.envs[0],
    DISCRETE_ACTIONS,
)

# Ensure VecEnv advertises the wrapped action space
h2_vec.action_space = h2_vec.envs[0].action_space

print("H2 action space :", h2_vec.action_space)
print("DQN action space:", dqn_tmp.action_space)
print("Action spaces match:",
      h2_vec.action_space == dqn_tmp.action_space)

H2 action space : Discrete(5)
DQN action space: Discrete(5)
Action spaces match: True


In [ ]:
# ------------------------------------------------------------
# Load frozen DQN + evaluate untouched 2025 H2
# ------------------------------------------------------------

model = DQN.load(
    MODEL_PATH,
    env=h2_vec,
    device="cuda",
)

h2_vec.training = False
h2_vec.norm_reward = False

obs = h2_vec.reset()

rewards = []
actions = []
done = False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = h2_vec.step(action)

    rewards.append(float(reward[0]))
    actions.append(int(action[0]))

total_reward = sum(rewards)

print("========================================")
print("DQN + TFT | 2025 H2 TEST")
print("========================================")
print(f"Steps        : {len(rewards)}")
print(f"Total reward : {total_reward:,.2f}")
print(f"Mean hourly  : {np.mean(rewards):,.2f}")
print(f"Min reward   : {np.min(rewards):,.2f}")
print(f"Max reward   : {np.max(rewards):,.2f}")
print(f"Actions      : {np.bincount(actions, minlength=5)}")
print("========================================")

DQN + TFT | 2025 H2 TEST
Steps        : 4416
Total reward : -12,262.93
Mean hourly  : -2.78
Min reward   : -378.01
Max reward   : 405.88
Actions      : [   0  150  103  393 3770]


In [ ]:
# ============================================================
# DQN + CAUSAL TFT — 20 EPISODES
# ============================================================

import os
import sys
import numpy as np
import torch
import torch.nn as nn

from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecNormalize

# ------------------------------------------------------------
# Repo
# ------------------------------------------------------------
REPO = "/content/XRL-for-optimising-battery-schedules"

if REPO not in sys.path:
    sys.path.insert(0, REPO)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
TRAIN_FILE = (
    f"{REPO}/data/rl/"
    "german_rl_combined_tft_2022_2025H1.csv"
)

SAVE_DIR = (
    f"{REPO}/log/causal_rl/"
    "dqn_tft/run_01_20ep"
)

os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------
EPISODES = 20
EPISODE_LENGTH = 21_960
TOTAL_TIMESTEPS = EPISODES * EPISODE_LENGTH

SEED = 22

DISCRETE_ACTIONS = [
    np.array([-1.0], dtype=np.float32),
    np.array([-0.5], dtype=np.float32),
    np.array([ 0.0], dtype=np.float32),
    np.array([ 0.5], dtype=np.float32),
    np.array([ 1.0], dtype=np.float32),
]

# ------------------------------------------------------------
# Create causal TFT environment
# ------------------------------------------------------------
from envs.env_params import de_bat_baseline

env_kwargs = dict(de_bat_baseline)
env_kwargs["data_file"] = TRAIN_FILE

env = CausalTFTBatteryEnv(
    **env_kwargs
)

# ------------------------------------------------------------
# Use the repo's normal environment wrapper pipeline
# ------------------------------------------------------------
from utils.make_env import make_env

env.close()

env = make_env(
    env=CausalTFTBatteryEnv,
    env_kwargs=env_kwargs,
    path=f"{SAVE_DIR}/train_monitor.csv",
    perfect_forecasts=None,
    forecasts=None,
    flatten_obs=True,
    discrete_actions=DISCRETE_ACTIONS,
    norm_obs=True,
    norm_reward=True,
    gamma=0.975431,
)

# ------------------------------------------------------------
# Verify state/action dimensions
# ------------------------------------------------------------
print("=" * 70)
print("DQN + CAUSAL TFT — 20 EPISODES")
print("=" * 70)
print(f"Training file       : {TRAIN_FILE}")
print(f"Observation dim     : {env.observation_space.shape}")
print(f"Action space        : {env.action_space}")
print(f"Episodes            : {EPISODES}")
print(f"Hours / episode     : {EPISODE_LENGTH:,}")
print(f"Total timesteps     : {TOTAL_TIMESTEPS:,}")
print(f"Seed                : {SEED}")
print(f"Output directory    : {SAVE_DIR}")
print("=" * 70)

assert env.observation_space.shape == (117,), (
    f"Expected 117-D observation, got "
    f"{env.observation_space.shape}"
)

assert env.action_space.n == 5, (
    f"Expected 5 actions, got {env.action_space}"
)

# ------------------------------------------------------------
# DQN
# ------------------------------------------------------------
model = DQN(
    policy="MlpPolicy",
    env=env,

    # Linear decay: 4.8e-4 -> 0
    learning_rate=lambda progress_remaining:
        4.8e-4 * progress_remaining,

    # Replay
    buffer_size=500_000,
    batch_size=256,
    learning_starts=2_429,

    # Updates
    train_freq=148,
    gradient_steps=-1,

    # Target network
    tau=0.435965,
    target_update_interval=10_000,

    # Discount
    gamma=0.975431,

    # Epsilon-greedy exploration
    exploration_fraction=0.50,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.01,

    # Gradient clipping
    max_grad_norm=0.467447,

    # Repo extra-large architecture
    # LeakyReLU activation
    policy_kwargs=dict(
        net_arch=[256, 512, 512, 256],
        activation_fn=nn.LeakyReLU,
    ),

    seed=SEED,
    verbose=1,
    device="cuda",
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
print("\nStarting training...\n")

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    progress_bar=True,
)

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------
MODEL_PATH = f"{SAVE_DIR}/model_final"

model.save(MODEL_PATH)

# ------------------------------------------------------------
# IMPORTANT:
# Save the exact observation/reward normalization statistics
# used during training.
# ------------------------------------------------------------
VECNORM_PATH = f"{SAVE_DIR}/vecnormalize.pkl"

env.save(VECNORM_PATH)

# ------------------------------------------------------------
# Finish
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("DQN + CAUSAL TFT TRAINING COMPLETE")
print("=" * 70)
print(f"Episodes completed : {EPISODES}")
print(f"Timesteps          : {model.num_timesteps:,}")
print(f"Model              : {MODEL_PATH}.zip")
print(f"VecNormalize       : {VECNORM_PATH}")
print("=" * 70)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

Output()

/usr/local/lib/python3.13/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

DQN + CAUSAL TFT — 20 EPISODES
Training file       : /content/XRL-for-optimising-battery-schedules/data/rl/german_rl_combined_tft_2022_2025H1.csv
Observation dim     : (117,)
Action space        : Discrete(5)
Episodes            : 20
Hours / episode     : 21,960
Total timesteps     : 439,200
Seed                : 22
Output directory    : /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_tft/run_01_20ep
Using cuda device

Starting training...

-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 2.63e+04  |
|    ep_rew_mean      | -2.25e+06 |
|    exploration_rate | 0.526     |
| time/               |           |
|    episodes         | 4         |
|    fps              | 228       |
|    time_elapsed     | 460       |
|    total_timesteps  | 105216    |
| train/              |           |
|    learning_rate    | 0.000365  |
|    loss             | 0.000122  |
|    n_updates        | 102712    |
-----------------------------------


DQN + CAUSAL TFT TRAINING COMPLETE
Episodes completed : 20
Timesteps          : 439,264
Model              : /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_tft/run_01_20ep/model_final.zip
VecNormalize       : /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_tft/run_01_20ep/vecnormalize.pkl


In [ ]:
# ============================================================
# EVALUATE DQN + CAUSAL TFT ON 2025 H1
# H2 REMAINS UNTOUCHED
# ============================================================

import os
import sys
import numpy as np
import pandas as pd

from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecNormalize

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
REPO = "/content/XRL-for-optimising-battery-schedules"
sys.path.insert(0, REPO)

MODEL_PATH = (
    f"{REPO}/log/causal_rl/dqn_tft/run_01_20ep/model_final.zip"
)
VECNORM_PATH = (
    f"{REPO}/log/causal_rl/dqn_tft/run_01_20ep/vecnormalize.pkl"
)

COMBINED_FILE = (
    f"{REPO}/data/rl/german_rl_combined_tft_2022_2025H1.csv"
)

H1_FILE = f"{REPO}/data/rl/_eval_dqn_tft_2025H1.csv"

# ------------------------------------------------------------
# Build H1 evaluation file from the causal TFT dataset
# ------------------------------------------------------------
df = pd.read_csv(COMBINED_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"])

h1 = df[
    (df["timestamp"] >= "2025-01-01 00:00:00") &
    (df["timestamp"] <= "2025-06-30 23:00:00")
].copy()

h1.to_csv(H1_FILE, index=False)

print("=" * 70)
print("DQN + CAUSAL TFT — 2025 H1 EVALUATION")
print("=" * 70)
print(f"H1 rows : {len(h1):,}")
print(f"Start   : {h1['timestamp'].min()}")
print(f"End     : {h1['timestamp'].max()}")
print(f"Model   : {MODEL_PATH}")
print("=" * 70)

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------
from envs.env_params import de_bat_baseline

env_kwargs = dict(de_bat_baseline)
env_kwargs["data_file"] = H1_FILE

# CausalTFTBatteryEnv was defined in the training cell
eval_env_raw = CausalTFTBatteryEnv(**env_kwargs)

# Use exactly the same wrapper stack as training
from utils.make_env import make_env

eval_env = make_env(
    env=CausalTFTBatteryEnv,
    env_kwargs=env_kwargs,
    flatten_obs=True,
    discrete_actions=[
        np.array([-1.0]),
        np.array([-0.5]),
        np.array([0.0]),
        np.array([0.5]),
        np.array([1.0]),
    ],
    norm_obs=True,
    norm_reward=False,
    gamma=0.975431,
)

# ------------------------------------------------------------
# Load TRAINING normalization statistics
# ------------------------------------------------------------
eval_env = VecNormalize.load(VECNORM_PATH, eval_env)

# Critical: freeze training statistics during evaluation
eval_env.training = False
eval_env.norm_reward = False

# ------------------------------------------------------------
# Load trained DQN
# ------------------------------------------------------------
model = DQN.load(
    MODEL_PATH,
    env=eval_env,
    device="cuda",
)

# ------------------------------------------------------------
# Run deterministic evaluation
# ------------------------------------------------------------
obs = eval_env.reset()

total_reward = 0.0
steps = 0
actions = []

while True:
    action, _ = model.predict(obs, deterministic=True)

    obs, reward, done, info = eval_env.step(action)

    total_reward += float(reward[0])
    steps += 1
    actions.append(int(action[0]))

    if done[0]:
        break

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
actions = np.array(actions)

print()
print("=" * 70)
print("2025 H1 EVALUATION COMPLETE")
print("=" * 70)
print(f"Steps             : {steps:,}")
print(f"Total reward      : {total_reward:,.2f}")
print(f"Mean reward/hour  : {total_reward / max(steps, 1):,.4f}")
print()
print("Action distribution:")
for a in range(5):
    n = np.sum(actions == a)
    print(f"  Action {a}: {n:5d} ({100*n/len(actions):6.2f}%)")

print()
print("Action mapping:")
print("  0 = -1.0  (charge)")
print("  1 = -0.5")
print("  2 =  0.0  (idle)")
print("  3 = +0.5")
print("  4 = +1.0  (discharge)")
print("=" * 70)

eval_env.close()
eval_env_raw.close()

DQN + CAUSAL TFT — 2025 H1 EVALUATION
H1 rows : 4,344
Start   : 2025-01-01 00:00:00
End     : 2025-06-30 23:00:00
Model   : /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_tft/run_01_20ep/model_final.zip


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



2025 H1 EVALUATION COMPLETE
Steps             : 4,344
Total reward      : -1,094.81
Mean reward/hour  : -0.2520

Action distribution:
  Action 0:    10 (  0.23%)
  Action 1:   585 ( 13.47%)
  Action 2:  1075 ( 24.75%)
  Action 3:  1960 ( 45.12%)
  Action 4:   714 ( 16.44%)

Action mapping:
  0 = -1.0  (charge)
  1 = -0.5
  2 =  0.0  (idle)
  3 = +0.5
  4 = +1.0  (discharge)


In [ ]:
# ============================================================
# FINAL TEST — DQN + CAUSAL TFT — 2025 H2
# H2 IS UNTOUCHED: NO TRAINING / NO FITTING
# ============================================================

import os
import sys
import numpy as np
import pandas as pd

from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecNormalize

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
REPO = "/content/XRL-for-optimising-battery-schedules"
sys.path.insert(0, REPO)

MODEL_PATH = (
    f"{REPO}/log/causal_rl/dqn_tft/run_01_20ep/model_final.zip"
)

VECNORM_PATH = (
    f"{REPO}/log/causal_rl/dqn_tft/run_01_20ep/vecnormalize.pkl"
)

H2_FILE = (
    f"{REPO}/data/rl/german_rl_h2_2025.csv"
)

# ------------------------------------------------------------
# Confirm untouched H2 test set
# ------------------------------------------------------------
h2 = pd.read_csv(H2_FILE)
h2["timestamp"] = pd.to_datetime(h2["timestamp"])

print("=" * 70)
print("FINAL TEST — DQN + CAUSAL TFT — 2025 H2")
print("=" * 70)
print(f"H2 rows : {len(h2):,}")
print(f"Start   : {h2['timestamp'].min()}")
print(f"End     : {h2['timestamp'].max()}")
print(f"Model   : {MODEL_PATH}")
print("=" * 70)

assert len(h2) == 4416, f"Unexpected H2 length: {len(h2)}"
assert h2["timestamp"].min() == pd.Timestamp("2025-07-01 00:00:00")
assert h2["timestamp"].max() == pd.Timestamp("2025-12-31 23:00:00")

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------
from envs.env_params import de_bat_baseline

env_kwargs = dict(de_bat_baseline)
env_kwargs["data_file"] = H2_FILE

from utils.make_env import make_env

discrete_actions = [
    np.array([-1.0]),
    np.array([-0.5]),
    np.array([0.0]),
    np.array([0.5]),
    np.array([1.0]),
]

eval_env = make_env(
    env=CausalTFTBatteryEnv,
    env_kwargs=env_kwargs,
    flatten_obs=True,
    discrete_actions=discrete_actions,
    norm_obs=True,
    norm_reward=False,
    gamma=0.975431,
)

# ------------------------------------------------------------
# Load EXACT training normalization
# ------------------------------------------------------------
eval_env = VecNormalize.load(VECNORM_PATH, eval_env)

# Freeze normalization — absolutely no updating on H2
eval_env.training = False
eval_env.norm_reward = False

# ------------------------------------------------------------
# Load trained DQN
# ------------------------------------------------------------
model = DQN.load(
    MODEL_PATH,
    env=eval_env,
    device="cuda",
)

# ------------------------------------------------------------
# Deterministic test evaluation
# ------------------------------------------------------------
obs = eval_env.reset()

total_reward = 0.0
steps = 0
actions = []

while True:
    action, _ = model.predict(obs, deterministic=True)

    obs, reward, done, info = eval_env.step(action)

    total_reward += float(reward[0])
    steps += 1
    actions.append(int(action[0]))

    if done[0]:
        break

actions = np.asarray(actions)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
print()
print("=" * 70)
print("FINAL H2 TEST RESULT")
print("=" * 70)
print(f"Test hours        : {steps:,}")
print(f"Total reward      : {total_reward:,.2f}")
print(f"Mean reward/hour  : {total_reward / steps:,.6f}")
print()

print("Action distribution:")
for a, value in enumerate([-1.0, -0.5, 0.0, 0.5, 1.0]):
    n = np.sum(actions == a)
    print(f"  {value:+4.1f}: {n:5d} ({100*n/len(actions):6.2f}%)")

print("=" * 70)
print("H2 TEST COMPLETE — MODEL WAS NOT UPDATED")
print("=" * 70)

eval_env.close()

FINAL TEST — DQN + CAUSAL TFT — 2025 H2
H2 rows : 4,416
Start   : 2025-07-01 00:00:00
End     : 2025-12-31 23:00:00
Model   : /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_tft/run_01_20ep/model_final.zip


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



FINAL H2 TEST RESULT
Test hours        : 4,416
Total reward      : -578.41
Mean reward/hour  : -0.130981

Action distribution:
  -1.0:   766 ( 17.35%)
  -0.5:   709 ( 16.06%)
  +0.0:  2343 ( 53.06%)
  +0.5:   414 (  9.38%)
  +1.0:   184 (  4.17%)
H2 TEST COMPLETE — MODEL WAS NOT UPDATED


In [ ]:
# ============================================================
# DQN BASELINE — 20 EPISODES
# State = SOC + 20 market features = 21 dimensions
# NO TFT FORECAST FEATURES
# ============================================================

import os
import sys
import numpy as np
import torch as th

REPO = "/content/XRL-for-optimising-battery-schedules"
sys.path.insert(0, REPO)

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_baseline
from utils.make_env import make_env
from stable_baselines3 import DQN
from utils.scheduler import linear_schedule

# ------------------------------------------------------------
# Training data
# IMPORTANT: same chronological training period as DQN+TFT
# ------------------------------------------------------------
TRAIN_FILE = (
    f"{REPO}/data/rl/german_rl_train_2022H2_2024.csv"
)

SAVE_DIR = (
    f"{REPO}/log/causal_rl/dqn_baseline/run_01_20ep"
)

os.makedirs(SAVE_DIR, exist_ok=True)

EPISODES = 20
EPISODE_LENGTH = 21_960
TOTAL_TIMESTEPS = EPISODES * EPISODE_LENGTH
SEED = 22

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------
env_kwargs = dict(de_bat_baseline)
env_kwargs["data_file"] = TRAIN_FILE

discrete_actions = [
    np.array([-1.0]),
    np.array([-0.5]),
    np.array([0.0]),
    np.array([0.5]),
    np.array([1.0]),
]

# ------------------------------------------------------------
# Build wrapped environment
# ------------------------------------------------------------
env = make_env(
    env=FreeBatteryEnv,
    env_kwargs=env_kwargs,
    flatten_obs=True,
    discrete_actions=discrete_actions,
    norm_obs=True,
    norm_reward=True,
    gamma=0.975431,
)

# ------------------------------------------------------------
# Confirm baseline state/action dimensions
# ------------------------------------------------------------
print("=" * 70)
print("DQN BASELINE — 20 EPISODES")
print("=" * 70)
print(f"Training file       : {TRAIN_FILE}")
print(f"Observation dim     : {env.observation_space.shape}")
print(f"Action space        : {env.action_space}")
print(f"Episodes            : {EPISODES}")
print(f"Hours / episode     : {EPISODE_LENGTH:,}")
print(f"Total timesteps     : {TOTAL_TIMESTEPS:,}")
print(f"Seed                : {SEED}")
print(f"Output directory    : {SAVE_DIR}")
print("=" * 70)

assert env.observation_space.shape == (21,), (
    f"Expected 21-dimensional baseline state, "
    f"got {env.observation_space.shape}"
)

assert env.action_space.n == 5

# ------------------------------------------------------------
# DQN hyperparameters
# Same setup used for the TFT model
# ------------------------------------------------------------
policy_kwargs = dict(
    net_arch=[256, 512, 512, 256],
    activation_fn=th.nn.LeakyReLU,
)

model = DQN(
    policy="MlpPolicy",
    env=env,

    learning_rate=linear_schedule(4.8e-4),

    buffer_size=500_000,
    batch_size=256,
    learning_starts=2_429,
    train_freq=148,
    gradient_steps=-1,

    tau=0.435965,
    target_update_interval=10_000,

    gamma=0.975431,

    exploration_fraction=0.50,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.01,

    max_grad_norm=0.467447,

    policy_kwargs=policy_kwargs,

    seed=SEED,
    verbose=1,
    device="cuda",
)

print("\nStarting baseline DQN training...\n")

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    progress_bar=True,
)

# ------------------------------------------------------------
# Save model + exact normalization statistics
# ------------------------------------------------------------
MODEL_PATH = f"{SAVE_DIR}/model_final.zip"
VECNORM_PATH = f"{SAVE_DIR}/vecnormalize.pkl"

model.save(MODEL_PATH)
env.save(VECNORM_PATH)

print()
print("=" * 70)
print("DQN BASELINE TRAINING COMPLETE")
print("=" * 70)
print(f"Episodes completed : {EPISODES}")
print(f"Timesteps          : {model.num_timesteps:,}")
print(f"Model              : {MODEL_PATH}")
print(f"VecNormalize       : {VECNORM_PATH}")
print("=" * 70)

env.close()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

Output()

/usr/local/lib/python3.13/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

DQN BASELINE — 20 EPISODES
Training file       : /content/XRL-for-optimising-battery-schedules/data/rl/german_rl_train_2022H2_2024.csv
Observation dim     : (21,)
Action space        : Discrete(5)
Episodes            : 20
Hours / episode     : 21,960
Total timesteps     : 439,200
Seed                : 22
Output directory    : /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_baseline/run_01_20ep
Using cuda device

Starting baseline DQN training...

-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 2.2e+04   |
|    ep_rew_mean      | -1.98e+06 |
|    exploration_rate | 0.604     |
| time/               |           |
|    episodes         | 4         |
|    fps              | 252       |
|    time_elapsed     | 348       |
|    total_timesteps  | 87840     |
| train/              |           |
|    learning_rate    | 0.000384  |
|    loss             | 0.000102  |
|    n_updates        | 85396     |
-----------------------------


DQN BASELINE TRAINING COMPLETE
Episodes completed : 20
Timesteps          : 439,264
Model              : /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_baseline/run_01_20ep/model_final.zip
VecNormalize       : /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_baseline/run_01_20ep/vecnormalize.pkl


In [ ]:
# ============================================================
# DQN BASELINE — 2025 H1 EVALUATION
# ============================================================

import os
import sys
import numpy as np
import pandas as pd

from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecNormalize

REPO = "/content/XRL-for-optimising-battery-schedules"
sys.path.insert(0, REPO)

MODEL_PATH = (
    f"{REPO}/log/causal_rl/dqn_baseline/run_01_20ep/model_final.zip"
)

VECNORM_PATH = (
    f"{REPO}/log/causal_rl/dqn_baseline/run_01_20ep/vecnormalize.pkl"
)

H1_FILE = (
    f"{REPO}/data/rl/german_rl_validation_2025H1.csv"
)

h1 = pd.read_csv(H1_FILE)
h1["timestamp"] = pd.to_datetime(h1["timestamp"])

print("=" * 70)
print("DQN BASELINE — 2025 H1 EVALUATION")
print("=" * 70)
print(f"H1 rows : {len(h1):,}")
print(f"Start   : {h1['timestamp'].min()}")
print(f"End     : {h1['timestamp'].max()}")
print(f"Model   : {MODEL_PATH}")
print("=" * 70)

assert len(h1) == 4344

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_baseline
from utils.make_env import make_env

env_kwargs = dict(de_bat_baseline)
env_kwargs["data_file"] = H1_FILE

discrete_actions = [
    np.array([-1.0]),
    np.array([-0.5]),
    np.array([0.0]),
    np.array([0.5]),
    np.array([1.0]),
]

eval_env = make_env(
    env=FreeBatteryEnv,
    env_kwargs=env_kwargs,
    flatten_obs=True,
    discrete_actions=discrete_actions,
    norm_obs=True,
    norm_reward=False,
    gamma=0.975431,
)

eval_env = VecNormalize.load(VECNORM_PATH, eval_env)

# Freeze training normalization
eval_env.training = False
eval_env.norm_reward = False

model = DQN.load(
    MODEL_PATH,
    env=eval_env,
    device="cuda",
)

obs = eval_env.reset()

total_reward = 0.0
steps = 0
actions = []

while True:
    action, _ = model.predict(obs, deterministic=True)

    obs, reward, done, info = eval_env.step(action)

    total_reward += float(reward[0])
    steps += 1
    actions.append(int(action[0]))

    if done[0]:
        break

actions = np.asarray(actions)

print()
print("=" * 70)
print("DQN BASELINE — H1 RESULT")
print("=" * 70)
print(f"Steps             : {steps:,}")
print(f"Total reward      : {total_reward:,.2f}")
print(f"Mean reward/hour  : {total_reward / steps:,.6f}")
print()
print("Action distribution:")

for a, value in enumerate([-1.0, -0.5, 0.0, 0.5, 1.0]):
    n = np.sum(actions == a)
    print(f"  {value:+4.1f}: {n:5d} ({100*n/len(actions):6.2f}%)")

print("=" * 70)

eval_env.close()

DQN BASELINE — 2025 H1 EVALUATION
H1 rows : 4,344
Start   : 2025-01-01 00:00:00
End     : 2025-06-30 23:00:00
Model   : /content/XRL-for-optimising-battery-schedules/log/causal_rl/dqn_baseline/run_01_20ep/model_final.zip


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



DQN BASELINE — H1 RESULT
Steps             : 4,344
Total reward      : -17,732.75
Mean reward/hour  : -4.082125

Action distribution:
  -1.0:   728 ( 16.76%)
  -0.5:   862 ( 19.84%)
  +0.0:  1152 ( 26.52%)
  +0.5:   495 ( 11.40%)
  +1.0:  1107 ( 25.48%)


In [ ]:
import sys
import os

REPO = "/content/XRL-for-optimising-battery-schedules"

# The repo's imports are "envs.*", "utils.*", etc.
sys.path.insert(0, os.path.join(REPO, "src"))

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_baseline

print("Import successful:", FreeBatteryEnv)
print("Baseline environment:", de_bat_baseline["env_name"])

Import successful: <class 'envs.environments.FreeBatteryEnv'>
Baseline environment: de_bat_baseline


In [ ]:
from utils.make_env import make_env

# Exact baseline configuration, with ONLY the data file changed to H2.
env_kwargs = dict(de_bat_baseline)
env_kwargs["data_file"] = H2_FILE

env = make_env(
    env=FreeBatteryEnv,
    env_kwargs=env_kwargs,
    flatten_obs=True,
    discrete_actions=[
        np.array([-1.0]),
        np.array([-0.5]),
        np.array([0.0]),
        np.array([0.5]),
        np.array([1.0]),
    ],
    norm_obs=True,
    norm_reward=False,
    gamma=0.975431,
)

# Replace the freshly created normalizer with the exact one saved during training.
env = VecNormalize.load(VECNORM_PATH, env)
env.training = False
env.norm_reward = False

model = DQN.load(MODEL_PATH, env=env, device="cuda")

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

obs = env.reset()

rewards = []
actions = []

for _ in range(len(h2)):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = env.step(action)

    rewards.append(float(reward[0]))
    actions.append(int(action[0]))

    if done[0]:
        break

rewards = np.asarray(rewards)
actions = np.asarray(actions)

action_values = [-1.0, -0.5, 0.0, 0.5, 1.0]

print("=" * 60)
print("DQN BASELINE — 2025 H2 FINAL TEST")
print("=" * 60)
print(f"Steps evaluated : {len(rewards)}")
print(f"Total reward    : {rewards.sum():,.2f}")
print(f"Mean reward/hr  : {rewards.mean():,.6f}")
print()
print("Action distribution:")

for i, value in enumerate(action_values):
    n = int((actions == i).sum())
    print(f"  {value:+.1f}: {n:4d} ({100*n/len(actions):6.2f}%)")

print("=" * 60)

Observation space: Box([ 0.000000e+00 -5.000000e+02 -5.000000e+02 -5.000000e+02 -5.000000e+02
 -5.000000e+02 -1.521800e+02 -2.501305e+01 -2.501305e+01 -5.000000e+02
  0.000000e+00  4.650000e+01  1.250000e+00  3.090275e+04 -8.323750e+03
 -1.000000e+00 -1.000000e+00 -9.749280e-01 -9.009690e-01 -1.000000e+00
 -1.000000e+00], [1.0000000e+00 9.3628003e+02 9.3628003e+02 9.3628003e+02 9.3628003e+02
 9.3628003e+02 8.1935999e+02 5.7547363e+02 5.7547363e+02 5.0000000e+02
 7.6332500e+03 4.8499500e+04 4.6897500e+04 7.5508250e+04 6.7957750e+04
 1.0000000e+00 1.0000000e+00 9.7492802e-01 1.0000000e+00 1.0000000e+00
 1.0000000e+00], (21,), float32)
Action space: Discrete(5)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DQN BASELINE — 2025 H2 FINAL TEST
Steps evaluated : 4416
Total reward    : -32,060.39
Mean reward/hr  : -7.260052

Action distribution:
  -1.0: 1008 ( 22.83%)
  -0.5:  377 (  8.54%)
  +0.0: 1176 ( 26.63%)
  +0.5:  341 (  7.72%)
  +1.0: 1514 ( 34.28%)


In [ ]:
import os
import sys
import numpy as np

sys.path.insert(0, "/content/XRL-for-optimising-battery-schedules/src")

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize

from utils.make_env import make_env
from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_ppo
from utils.net_design import activation_fn_dict
from utils.scheduler import linear_schedule

# ============================================================
# PPO BASELINE — FINAL 20-EPISODE TRAINING
# ============================================================

TRAIN_FILE = (
    "/content/XRL-for-optimising-battery-schedules/"
    "data/rl/german_rl_train_2022H2_2024.csv"
)

OUT_DIR = (
    "/content/XRL-for-optimising-battery-schedules/"
    "log/causal_rl/ppo_baseline_20ep/run_01"
)

os.makedirs(OUT_DIR, exist_ok=True)

EPISODES = 20
EPISODE_LENGTH = 21_960
TOTAL_TIMESTEPS = EPISODES * EPISODE_LENGTH

# Exact PPO environment configuration
env_kwargs = dict(de_bat_ppo)
env_kwargs["data_file"] = TRAIN_FILE

env = make_env(
    env=FreeBatteryEnv,
    env_kwargs=env_kwargs,
    flatten_obs=True,
    discrete_actions=None,      # PPO is continuous
    norm_obs=True,
    norm_reward=True,
    gamma=0.900887,
)

print("=" * 70)
print("PPO BASELINE — FINAL 20-EPISODE TRAINING")
print("=" * 70)
print(f"Training file : {TRAIN_FILE}")
print(f"Observation   : {env.observation_space}")
print(f"Action space  : {env.action_space}")
print(f"Episodes      : {EPISODES}")
print(f"Hours/episode : {EPISODE_LENGTH:,}")
print(f"Timesteps     : {TOTAL_TIMESTEPS:,}")
print("=" * 70)

# Repository PPO configuration
policy_kwargs = {
    "net_arch": [128, 128],
    "activation_fn": activation_fn_dict["tanh"],
    "ortho_init": False,
    "squash_output": True,
    "share_features_extractor": True,
}

model = PPO(
    "MlpPolicy",
    env,
    learning_rate=linear_schedule(1.1e-5),
    n_steps=1024,
    batch_size=128,
    n_epochs=5,
    gamma=0.900887,
    gae_lambda=0.985039,
    clip_range=linear_schedule(0.176607),
    clip_range_vf=None,
    normalize_advantage=False,
    ent_coef=0.969954,
    vf_coef=0.696368,
    max_grad_norm=0.411048,
    use_sde=True,
    target_kl=None,
    policy_kwargs=policy_kwargs,
    seed=22,
    verbose=1,
    device="cuda",
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
model.learn(total_timesteps=TOTAL_TIMESTEPS)

# ------------------------------------------------------------
# Save model + EXACT training normalization statistics
# ------------------------------------------------------------
MODEL_PATH = os.path.join(OUT_DIR, "model_final")
VECNORM_PATH = os.path.join(OUT_DIR, "vecnormalize.pkl")

model.save(MODEL_PATH)
env.save(VECNORM_PATH)

print()
print("=" * 70)
print("PPO BASELINE — TRAINING COMPLETE")
print("=" * 70)
print(f"Episodes requested : {EPISODES}")
print(f"Timesteps requested: {TOTAL_TIMESTEPS:,}")
print(f"Timesteps reached  : {model.num_timesteps:,}")
print(f"Model              : {MODEL_PATH}.zip")
print(f"VecNormalize       : {VECNORM_PATH}")
print("=" * 70)

PPO BASELINE — FINAL 20-EPISODE TRAINING
Training file : /content/XRL-for-optimising-battery-schedules/data/rl/german_rl_train_2022H2_2024.csv
Observation   : Box([ 0.000000e+00 -5.000000e+02 -5.000000e+02 -5.000000e+02 -5.000000e+02
 -5.000000e+02 -1.521800e+02 -2.501305e+01 -2.501305e+01 -5.000000e+02
  0.000000e+00  4.650000e+01  1.250000e+00  3.090275e+04 -8.323750e+03
 -1.000000e+00 -1.000000e+00 -9.749280e-01 -9.009690e-01 -1.000000e+00
 -1.000000e+00], [1.0000000e+00 9.3628003e+02 9.3628003e+02 9.3628003e+02 9.3628003e+02
 9.3628003e+02 8.1935999e+02 5.7547363e+02 5.7547363e+02 5.0000000e+02
 7.6332500e+03 4.8499500e+04 4.6897500e+04 7.5508250e+04 6.7957750e+04
 1.0000000e+00 1.0000000e+00 9.7492802e-01 1.0000000e+00 1.0000000e+00
 1.0000000e+00], (21,), float32)
Action space  : Box(-1.0, 1.0, (1,), float32)
Episodes      : 20
Hours/episode : 21,960
Timesteps     : 439,200
Using cuda device


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Streaming output truncated to the last 5000 lines.
| time/                   |              |
|    fps                  | 440          |
|    iterations           | 203          |
|    time_elapsed         | 471          |
|    total_timesteps      | 207872       |
| train/                  |              |
|    approx_kl            | 0.0016148756 |
|    clip_fraction        | 0.0361       |
|    clip_range           | 0.0934       |
|    entropy_loss         | 4.78         |
|    explained_variance   | 0.574        |
|    learning_rate        | 5.82e-06     |
|    loss                 | 7.4          |
|    n_updates            | 1010         |
|    policy_gradient_loss | 1.35         |
|    std                  | 1.02         |
|    value_loss           | 1.88         |
------------------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 2.2e+04      |
|    ep_rew_mean          | -1.6e+06     |
| t

In [ ]:
# ============================================================
# PPO BASELINE — FINAL H2 TEST EVALUATION
# 2025-07-01 00:00 -> 2025-12-31 23:00
# ============================================================

import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, "/content/XRL-for-optimising-battery-schedules")

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_ppo
from gymnasium.wrappers import FlattenObservation

H2_FILE = "/content/XRL-for-optimising-battery-schedules/data/rl/german_rl_h2_2025.csv"

MODEL_FILE = (
    "/content/XRL-for-optimising-battery-schedules/"
    "log/causal_rl/ppo_baseline_20ep/run_01/model_final.zip"
)

VECNORM_FILE = (
    "/content/XRL-for-optimising-battery-schedules/"
    "log/causal_rl/ppo_baseline_20ep/run_01/vecnormalize.pkl"
)

# ---------- Verify untouched H2 test period ----------
h2 = pd.read_csv(H2_FILE)
h2["timestamp"] = pd.to_datetime(h2["timestamp"])

assert len(h2) == 4416
assert h2["timestamp"].min() == pd.Timestamp("2025-07-01 00:00:00")
assert h2["timestamp"].max() == pd.Timestamp("2025-12-31 23:00:00")

print("=" * 70)
print("PPO BASELINE — FINAL H2 TEST")
print("=" * 70)
print(f"Test rows : {len(h2):,}")
print(f"Start     : {h2['timestamp'].min()}")
print(f"End       : {h2['timestamp'].max()}")

# ---------- Create the SAME baseline environment ----------
def make_h2_env():
    env_params = dict(de_bat_ppo)
    env_params["data_file"] = H2_FILE

    env = FreeBatteryEnv(**env_params)
    env = FlattenObservation(env)
    return env

vec_env = DummyVecEnv([make_h2_env])

# Load the normalization statistics learned during training.
vec_env = VecNormalize.load(VECNORM_FILE, vec_env)

# CRITICAL: frozen normalization for final test.
vec_env.training = False
vec_env.norm_reward = False

# ---------- Load final PPO ----------
model = PPO.load(
    MODEL_FILE,
    env=vec_env,
    device="cuda",
)

print(f"Observation dimension : {model.observation_space.shape}")
print(f"Action space          : {model.action_space}")
print("VecNormalize          : loaded from training")
print("Normalization frozen  : True")
print("Reward normalization  : False")

# ---------- Deterministic H2 rollout ----------
obs = vec_env.reset()

total_reward = 0.0
actions = []
rewards = []

for step in range(len(h2)):
    action, _ = model.predict(obs, deterministic=True)

    obs, reward, done, info = vec_env.step(action)

    r = float(reward[0])
    total_reward += r
    rewards.append(r)
    actions.append(float(np.asarray(action).reshape(-1)[0]))

    if done[0] and step < len(h2) - 1:
        raise RuntimeError(
            f"Environment terminated unexpectedly at H2 step {step}"
        )

vec_env.close()

rewards = np.asarray(rewards)
actions = np.asarray(actions)

# ---------- Results ----------
mean_reward = rewards.mean()

action_bins = {
    "-1.0": np.sum(actions <= -0.75),
    "-0.5": np.sum((actions > -0.75) & (actions <= -0.25)),
    "0":    np.sum((actions > -0.25) & (actions < 0.25)),
    "+0.5": np.sum((actions >= 0.25) & (actions < 0.75)),
    "+1.0": np.sum(actions >= 0.75),
}

print("\n" + "=" * 70)
print("PPO BASELINE — H2 FINAL TEST RESULTS")
print("=" * 70)
print(f"Total reward       : {total_reward:,.2f}")
print(f"Mean reward / hour : {mean_reward:,.6f}")
print(f"Min action         : {actions.min():.4f}")
print(f"Max action         : {actions.max():.4f}")
print(f"Mean action        : {actions.mean():.4f}")

print("\nAction distribution:")
for action, count in action_bins.items():
    print(f"  {action:>4} : {count:4d} ({100*count/len(actions):6.2f}%)")

print("=" * 70)

PPO BASELINE — FINAL H2 TEST
Test rows : 4,416
Start     : 2025-07-01 00:00:00
End       : 2025-12-31 23:00:00
Observation dimension : (21,)
Action space          : Box(-1.0, 1.0, (1,), float32)
VecNormalize          : loaded from training
Normalization frozen  : True
Reward normalization  : False


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



PPO BASELINE — H2 FINAL TEST RESULTS
Total reward       : -389,458.63
Mean reward / hour : -88.192624
Min action         : -0.9984
Max action         : 0.9979
Mean action        : 0.3799

Action distribution:
  -1.0 :  695 ( 15.74%)
  -0.5 :  289 (  6.54%)
     0 :  439 (  9.94%)
  +0.5 :  793 ( 17.96%)
  +1.0 : 2200 ( 49.82%)


In [15]:
# ============================================================
# PPO + TFT — 20 EPISODE TRAINING
# ============================================================

import os
import sys
import numpy as np
import pandas as pd
import torch

from gymnasium import spaces
from gymnasium.wrappers import FlattenObservation
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

sys.path.insert(0, "/content/XRL-for-optimising-battery-schedules")

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_ppo


# ============================================================
# PATHS
# ============================================================

TRAIN_FILE = (
    "/content/XRL-for-optimising-battery-schedules/"
    "data/rl/german_rl_combined_tft_2022_2025H1.csv"
)

SAVE_DIR = (
    "/content/XRL-for-optimising-battery-schedules/"
    "log/causal_rl/ppo_tft/run_01"
)

os.makedirs(SAVE_DIR, exist_ok=True)


# ============================================================
# LOAD TRAINING DATA
# ============================================================

train_df = pd.read_csv(TRAIN_FILE)

assert len(train_df) == 26304

# ============================================================
# EXACT 96 TFT FEATURES
# ============================================================

TFT_COLS = (
    [f"TFT_p10_h{i:02d}" for i in range(24)]
    + [f"TFT_p50_h{i:02d}" for i in range(24)]
    + [f"TFT_p90_h{i:02d}" for i in range(24)]
    + [f"TFT_uncertainty_h{i:02d}" for i in range(24)]
)

assert len(TFT_COLS) == 96

missing = [c for c in TFT_COLS if c not in train_df.columns]

assert not missing, f"Missing TFT columns: {missing}"


print("=" * 70)
print("PPO + TFT — TRAINING")
print("=" * 70)
print(f"Training rows : {len(train_df):,}")
print(f"Start         : {train_df['timestamp'].iloc[0]}")
print(f"End           : {train_df['timestamp'].iloc[-1]}")
print(f"TFT features  : {len(TFT_COLS)}")
print("Episodes      : 20")
print("Observation   : 117-D")
print("Action space  : continuous [-1, 1]")


# ============================================================
# CAUSAL TFT ENVIRONMENT
# ============================================================

class CausalTFTBatteryEnv(FreeBatteryEnv):

    def __init__(self, *args, **kwargs):

        # Disable the repo's built-in TFT feature.
        kwargs["tft_24h"] = None

        super().__init__(*args, **kwargs)

        # Load the exact 96 TFT columns from the RL dataset.
        tft_df = pd.read_csv(self.data_file)

        self.tft_data = (
            tft_df[TFT_COLS]
            .to_numpy(dtype=np.float32)
        )

        if len(self.tft_data) != len(self.data):

            raise ValueError(
                f"TFT rows ({len(self.tft_data)}) "
                f"!= environment rows ({len(self.data)})"
            )

        if not np.isfinite(self.tft_data).all():

            raise ValueError(
                "Non-finite values found in TFT features."
            )

        # 96 TFT features
        self.observation_space["tft_forecasts"] = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(96,),
            dtype=np.float32,
        )

    def _get_obs(self):

        obs, done = super()._get_obs()

        idx = min(
            self.count,
            len(self.tft_data) - 1
        )

        obs["tft_forecasts"] = self.tft_data[idx]

        return obs, done


# ============================================================
# ENVIRONMENT FACTORY
# ============================================================

def make_env():

    env_params = dict(de_bat_ppo)

    # Force the correct training dataset.
    env_params["data_file"] = TRAIN_FILE

    env = CausalTFTBatteryEnv(**env_params)

    env = Monitor(env)

    env = FlattenObservation(env)

    return env


vec_env = DummyVecEnv([make_env])


print()
print("Observation space:", vec_env.observation_space)
print("Action space     :", vec_env.action_space)

assert vec_env.observation_space.shape == (117,)
assert vec_env.action_space.shape == (1,)


# ============================================================
# NORMALIZATION
# ============================================================

vec_env = VecNormalize(
    vec_env,
    norm_obs=True,
    norm_reward=True,
    clip_obs=10.0,
)


# ============================================================
# PPO CONFIGURATION
# SAME AS PPO BASELINE
# ============================================================

def linear_schedule(initial_value):

    def func(progress_remaining):
        return progress_remaining * initial_value

    return func


policy_kwargs = dict(
    net_arch=[128, 128],
    ortho_init=False,
)


model = PPO(
    "MlpPolicy",
    vec_env,

    learning_rate=linear_schedule(1.1e-5),

    n_steps=1024,
    batch_size=128,
    n_epochs=5,

    gamma=0.900887,
    gae_lambda=0.985039,

    clip_range=linear_schedule(0.176607),

    normalize_advantage=False,

    ent_coef=0.969954,
    vf_coef=0.696368,

    max_grad_norm=0.411048,

    policy_kwargs=policy_kwargs,

    verbose=0,

    seed=22,

    device="cuda",
)


# ============================================================
# EPISODE REWARD CALLBACK
# ============================================================

class EpisodeRewardCallback(BaseCallback):

    def __init__(self):

        super().__init__()

        self.episode_rewards = []

    def _on_step(self):

        infos = self.locals.get("infos", [])

        for info in infos:

            if "episode" in info:

                reward = float(
                    info["episode"]["r"]
                )

                self.episode_rewards.append(
                    reward
                )

                episode = len(
                    self.episode_rewards
                )

                print(
                    f"Episode {episode:2d}/20 | "
                    f"Reward: {reward:,.2f} | "
                    f"Timesteps: {self.num_timesteps:,}"
                )

        return True


callback = EpisodeRewardCallback()


# ============================================================
# TRAIN
# ============================================================

TIMESTEPS = 20 * len(train_df)

print()
print("=" * 70)
print("PPO + TFT — STARTING TRAINING")
print("=" * 70)
print(f"Total timesteps : {TIMESTEPS:,}")
print(f"Seed            : 22")
print(f"Device          : {model.device}")
print("=" * 70)

model.learn(
    total_timesteps=TIMESTEPS,
    callback=callback,
    reset_num_timesteps=False,
)


# ============================================================
# SAVE MODEL + VECNORMALIZE
# ============================================================

MODEL_PATH = os.path.join(
    SAVE_DIR,
    "model_final",
)

VECNORM_PATH = os.path.join(
    SAVE_DIR,
    "vecnormalize.pkl",
)

model.save(MODEL_PATH)

vec_env.save(VECNORM_PATH)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print("=" * 70)
print("PPO + TFT TRAINING COMPLETE")
print("=" * 70)

print(f"Timesteps : {model.num_timesteps:,}")
print(f"Model     : {MODEL_PATH}.zip")
print(f"VecNorm   : {VECNORM_PATH}")

print()
print("Episode rewards:")

for i, reward in enumerate(
    callback.episode_rewards,
    start=1
):

    print(
        f"  Episode {i:2d}: {reward:,.2f}"
    )

print("=" * 70)

PPO + TFT — TRAINING
Training rows : 26,304
Start         : 2022-07-01 00:00:00
End           : 2025-06-30 23:00:00
TFT features  : 96
Episodes      : 20
Observation   : 117-D
Action space  : continuous [-1, 1]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Observation space: Box([ 0.000000e+00 -5.000000e+02 -5.000000e+02 -5.000000e+02 -5.000000e+02
 -5.000000e+02 -1.521800e+02 -2.501305e+01 -2.501305e+01 -5.000000e+02
  0.000000e+00  4.650000e+01  1.250000e+00  3.090275e+04 -8.443140e+03
 -1.000000e+00 -1.000000e+00 -9.749280e-01 -9.009690e-01 -1.000000e+00
 -1.000000e+00          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -

/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Episode  1/20 | Reward: -2,404,466.60 | Timesteps: 26,304
Episode  2/20 | Reward: -933,411.32 | Timesteps: 52,608
Episode  3/20 | Reward: -543,595.41 | Timesteps: 78,912
Episode  4/20 | Reward: -331,456.00 | Timesteps: 105,216
Episode  5/20 | Reward: -259,780.68 | Timesteps: 131,520
Episode  6/20 | Reward: -199,455.30 | Timesteps: 157,824
Episode  7/20 | Reward: -171,968.39 | Timesteps: 184,128
Episode  8/20 | Reward: -130,670.11 | Timesteps: 210,432
Episode  9/20 | Reward: -117,801.77 | Timesteps: 236,736
Episode 10/20 | Reward: -122,304.48 | Timesteps: 263,040
Episode 11/20 | Reward: -93,721.86 | Timesteps: 289,344
Episode 12/20 | Reward: -106,187.31 | Timesteps: 315,648
Episode 13/20 | Reward: -83,908.20 | Timesteps: 341,952
Episode 14/20 | Reward: -81,588.03 | Timesteps: 368,256
Episode 15/20 | Reward: -75,948.81 | Timesteps: 394,560
Episode 16/20 | Reward: -72,859.98 | Timesteps: 420,864
Episode 17/20 | Reward: -78,862.28 | Timesteps: 447,168
Episode 18/20 | Reward: -72,030.18 | T

In [18]:
# ============================================================
# PPO + TFT — FINAL H2 2025 TEST
# ============================================================

import os
import sys
import numpy as np
import pandas as pd

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from gymnasium.wrappers import FlattenObservation

# Re-add repo after runtime restart if necessary
REPO = "/content/XRL-for-optimising-battery-schedules"
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_ppo


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
H2_FILE = os.path.join(
    REPO,
    "data/rl/german_rl_h2_2025.csv"
)

MODEL_FILE = os.path.join(
    REPO,
    "log/causal_rl/ppo_tft/run_01/model_final.zip"
)

VECNORM_FILE = os.path.join(
    REPO,
    "log/causal_rl/ppo_tft/run_01/vecnormalize.pkl"
)


# ------------------------------------------------------------
# Exact TFT feature structure used during training
# ------------------------------------------------------------
TFT_COLS = (
    [f"TFT_p10_h{i:02d}" for i in range(24)]
    + [f"TFT_p50_h{i:02d}" for i in range(24)]
    + [f"TFT_p90_h{i:02d}" for i in range(24)]
    + [f"TFT_uncertainty_h{i:02d}" for i in range(24)]
)


# ------------------------------------------------------------
# Causal TFT environment
# ------------------------------------------------------------
class CausalTFTBatteryEnv(FreeBatteryEnv):

    def __init__(self, *args, **kwargs):
        kwargs["tft_24h"] = None
        super().__init__(*args, **kwargs)

        tft_df = pd.read_csv(self.data_file)

        missing = [c for c in TFT_COLS if c not in tft_df.columns]
        if missing:
            raise ValueError(f"Missing TFT columns: {missing}")

        self.tft_data = tft_df[TFT_COLS].to_numpy(dtype=np.float32)

        if len(self.tft_data) != len(self.data):
            raise ValueError(
                f"TFT rows ({len(self.tft_data)}) != env rows ({len(self.data)})"
            )

        if not np.isfinite(self.tft_data).all():
            raise ValueError("Non-finite TFT values detected.")

        # Same 96-dimensional TFT observation block used in training
        from gymnasium import spaces

        self.observation_space["tft_forecasts"] = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(96,),
            dtype=np.float32
        )

    def _get_obs(self):
        obs, done = super()._get_obs()

        idx = min(self.count, len(self.tft_data) - 1)
        obs["tft_forecasts"] = self.tft_data[idx]

        return obs, done


# ------------------------------------------------------------
# Verify H2 test data
# ------------------------------------------------------------
h2 = pd.read_csv(H2_FILE)

print("=" * 70)
print("PPO + TFT — FINAL H2 2025 TEST")
print("=" * 70)

print(f"Rows        : {len(h2):,}")
print(f"Start       : {h2['timestamp'].iloc[0]}")
print(f"End         : {h2['timestamp'].iloc[-1]}")
print(f"TFT features: {len(TFT_COLS)}")

assert len(h2) == 4416, f"Expected 4416 H2 rows, got {len(h2)}"


# ------------------------------------------------------------
# Build EXACT evaluation environment
# ------------------------------------------------------------
env_params = dict(de_bat_ppo)
env_params["data_file"] = H2_FILE

env = CausalTFTBatteryEnv(**env_params)
env = Monitor(env)
env = FlattenObservation(env)

vec_env = DummyVecEnv([lambda: env])

# Load training normalization statistics.
# IMPORTANT: freeze them for final testing.
vec_env = VecNormalize.load(VECNORM_FILE, vec_env)
vec_env.training = False
vec_env.norm_reward = False


print(f"Observation : {vec_env.observation_space}")
print(f"Action      : {vec_env.action_space}")


# ------------------------------------------------------------
# Load trained PPO + TFT model
# ------------------------------------------------------------
model = PPO.load(
    MODEL_FILE,
    env=vec_env,
    device="cuda"
)


# ------------------------------------------------------------
# Deterministic final evaluation
# ------------------------------------------------------------
obs = vec_env.reset()

rewards = []
actions = []

done = False

while not done:

    action, _ = model.predict(
        obs,
        deterministic=True
    )

    obs, reward, done_arr, info = vec_env.step(action)

    rewards.append(float(reward[0]))
    actions.append(float(np.asarray(action).reshape(-1)[0]))

    done = bool(done_arr[0])


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
rewards = np.asarray(rewards, dtype=np.float64)
actions = np.asarray(actions, dtype=np.float64)

total_reward = rewards.sum()
mean_reward = rewards.mean()

print()
print("=" * 70)
print("PPO + TFT — H2 FINAL TEST RESULTS")
print("=" * 70)

print(f"Test steps          : {len(rewards):,}")
print(f"Total reward        : {total_reward:,.2f}")
print(f"Mean reward / hour  : {mean_reward:,.6f}")

print()
print("Continuous actions:")
print(f"Minimum             : {actions.min():.4f}")
print(f"Maximum             : {actions.max():.4f}")
print(f"Mean                : {actions.mean():.4f}")

# Comparable action bins
bins = [-1.000001, -0.75, -0.25, 0.25, 0.75, 1.000001]
labels = ["-1.0", "-0.5", "0", "+0.5", "+1.0"]

action_bins = pd.cut(
    actions,
    bins=bins,
    labels=labels,
    include_lowest=True
)

counts = action_bins.value_counts().reindex(labels, fill_value=0)

print()
print("Action distribution:")
for label, count in counts.items():
    print(
        f"  {label:>4}: {count:4d} "
        f"({100 * count / len(actions):6.2f}%)"
    )

print()
print("=" * 70)
print("PPO + TFT H2 TEST COMPLETE")
print("=" * 70)

PPO + TFT — FINAL H2 2025 TEST
Rows        : 4,416
Start       : 2025-07-01 00:00:00
End         : 2025-12-31 23:00:00
TFT features: 96
Observation : Box([ 0.000000e+00 -5.000000e+02 -5.000000e+02 -5.000000e+02 -5.000000e+02
 -5.000000e+02 -1.521800e+02 -2.501305e+01 -2.501305e+01 -5.000000e+02
  0.000000e+00  4.650000e+01  1.250000e+00  3.090275e+04 -8.443140e+03
 -1.000000e+00 -1.000000e+00 -9.749280e-01 -9.009690e-01 -1.000000e+00
 -1.000000e+00          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf          -inf
          -inf          -inf          -inf          -inf         

/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



PPO + TFT — H2 FINAL TEST RESULTS
Test steps          : 4,416
Total reward        : -4,136.46
Mean reward / hour  : -0.936698

Continuous actions:
Minimum             : -1.0000
Maximum             : 1.0000
Mean                : -0.7547

Action distribution:
  -1.0: 3822 ( 86.55%)
  -0.5:   43 (  0.97%)
     0:   27 (  0.61%)
  +0.5:   34 (  0.77%)
  +1.0:  490 ( 11.10%)

PPO + TFT H2 TEST COMPLETE


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
# ============================================================
# SAC BASELINE — 20 EPISODE TRAINING
# Based on the SAC implementation/configuration in Demo2_vF(3)
# ============================================================

import os
import sys
import copy
import numpy as np
import pandas as pd
import torch

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.logger import configure

# ------------------------------------------------------------
# Repo
# ------------------------------------------------------------
REPO = "/content/XRL-for-optimising-battery-schedules"

if REPO not in sys.path:
    sys.path.insert(0, REPO)

# ------------------------------------------------------------
# Repository imports — same architecture as the notebook/repo
# ------------------------------------------------------------
from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_ppo
from utils.make_env import make_env


# ============================================================
# 1. PATHS
# ============================================================

TRAIN_FILE = os.path.join(
    REPO,
    "data",
    "rl",
    "german_rl_train_2022H2_2024.csv"
)

OUT_DIR = os.path.join(
    REPO,
    "log",
    "causal_rl",
    "sac_baseline_20ep",
    "run_01"
)

os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 2. TRAINING BUDGET
# ============================================================

EPISODE_LENGTH = 21_960
N_EPISODES = 20

TOTAL_TIMESTEPS = (
    EPISODE_LENGTH * N_EPISODES
)


# ============================================================
# 3. ENVIRONMENT CONFIGURATION
#
# Same German baseline environment pattern used for PPO.
# We override data_file explicitly for the actual RL train file.
# ============================================================

env_kwargs = copy.deepcopy(de_bat_ppo)

env_kwargs["data_file"] = TRAIN_FILE
env_kwargs["modeling_period_h"] = EPISODE_LENGTH


# ============================================================
# 4. SAC PARAMETERS
#
# From the SAC configuration in the uploaded notebook /
# Table C.8 — DE-LF/REU best configuration.
# ============================================================

SAC_PARAMS = {

    "learning_rate": 1e-4,

    "batch_size": 512,

    "buffer_size": 500_000,

    "tau": 0.006985,

    "target_update_interval": 5,

    "gamma": 0.999,

    "gradient_steps": -1,

    "learning_starts": 1_000,

    "train_freq": 35,

    "ent_coef": "auto_0.1",

    "target_entropy": "auto",

    "policy_kwargs": {
        "net_arch": [256, 512, 512, 256],
        "activation_fn": torch.nn.Tanh,
        "n_critics": 2,
        "share_features_extractor": True,
    },
}


# ============================================================
# 5. CREATE ENVIRONMENT
#
# This follows the notebook/repository make_env pattern:
#
# FreeBatteryEnv
#      ↓
# Monitor
#      ↓
# FlattenObservation
#      ↓
# DummyVecEnv
#      ↓
# VecNormalize
# ============================================================

env = make_env(
    env=FreeBatteryEnv,
    env_kwargs=env_kwargs,

    flatten_obs=True,

    norm_obs=True,
    norm_reward=True,

    # Important: SAC gamma is also passed to VecNormalize
    gamma=SAC_PARAMS["gamma"],

    # SAC uses continuous actions
    discrete_actions=None,
)


# ============================================================
# 6. VERIFY ENVIRONMENT
# ============================================================

print("=" * 70)
print("SAC BASELINE — TRAINING")
print("=" * 70)

print(f"Training file       : {TRAIN_FILE}")

train_df = pd.read_csv(TRAIN_FILE)

print(f"Training rows       : {len(train_df):,}")
print(f"Start               : {train_df['timestamp'].iloc[0]}")
print(f"End                 : {train_df['timestamp'].iloc[-1]}")

print(f"Observation space   : {env.observation_space}")
print(f"Observation shape   : {env.observation_space.shape}")
print(f"Action space        : {env.action_space}")

print(f"Episodes            : {N_EPISODES}")
print(f"Hours / episode     : {EPISODE_LENGTH:,}")
print(f"Timesteps           : {TOTAL_TIMESTEPS:,}")

assert len(train_df) == 21_960, (
    f"Expected 21,960 training rows, got {len(train_df)}"
)

assert env.observation_space.shape == (21,), (
    f"Expected 21-D baseline observation, "
    f"got {env.observation_space.shape}"
)

print("=" * 70)


# ============================================================
# 7. EPISODE REWARD CALLBACK
# ============================================================

class SACEpisodeCallback(BaseCallback):

    def __init__(self, verbose=0):
        super().__init__(verbose)

        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self):

        infos = self.locals.get("infos", [])

        for info in infos:

            episode_info = info.get("episode")

            if episode_info is not None:

                reward = float(episode_info["r"])
                length = int(episode_info["l"])

                self.episode_rewards.append(reward)
                self.episode_lengths.append(length)

                episode_number = len(
                    self.episode_rewards
                )

                print(
                    f"Episode "
                    f"{episode_number:2d}/{N_EPISODES} "
                    f"| Reward: {reward:,.2f} "
                    f"| Timesteps: "
                    f"{episode_number * EPISODE_LENGTH:,}"
                )

        return True


# ============================================================
# 8. CREATE SAC MODEL
# ============================================================

model = SAC(
    policy="MlpPolicy",
    env=env,

    # Exact SAC parameters
    **SAC_PARAMS,

    # Reproducibility
    seed=22,

    # A100
    device="cuda",

    # Keep SB3 output manageable
    verbose=0,
)


print()
print("=" * 70)
print("SAC BASELINE — STARTING TRAINING")
print("=" * 70)

print(f"Total timesteps : {TOTAL_TIMESTEPS:,}")
print(f"Seed            : 22")
print(f"Device          : {model.device}")
print(f"Observation     : {env.observation_space.shape[0]}-D")
print(f"Action space    : continuous [-1, 1]")
print("=" * 70)


# ============================================================
# 9. TRAIN
# ============================================================

callback = SACEpisodeCallback()

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callback,
    reset_num_timesteps=True,
    progress_bar=True,
)


# ============================================================
# 10. SAVE MODEL
# ============================================================

MODEL_PATH = os.path.join(
    OUT_DIR,
    "model_final"
)

model.save(MODEL_PATH)


# ============================================================
# 11. SAVE VECNORMALIZE
# ============================================================

VECNORM_PATH = os.path.join(
    OUT_DIR,
    "vecnormalize.pkl"
)

env.save(VECNORM_PATH)


# ============================================================
# 12. SAVE EPISODE REWARDS
# ============================================================

REWARD_FILE = os.path.join(
    OUT_DIR,
    "episode_rewards.csv"
)

reward_df = pd.DataFrame({
    "episode": np.arange(
        1,
        len(callback.episode_rewards) + 1
    ),
    "reward": callback.episode_rewards,
    "length": callback.episode_lengths,
})

reward_df.to_csv(
    REWARD_FILE,
    index=False
)


# ============================================================
# 13. FINAL REPORT
# ============================================================

print()
print("=" * 70)
print("SAC BASELINE — TRAINING COMPLETE")
print("=" * 70)

print(
    f"Episodes completed : "
    f"{len(callback.episode_rewards)} / {N_EPISODES}"
)

print(
    f"Timesteps requested: "
    f"{TOTAL_TIMESTEPS:,}"
)

print(
    f"Timesteps reached  : "
    f"{model.num_timesteps:,}"
)

print(
    f"Model              : "
    f"{MODEL_PATH}.zip"
)

print(
    f"VecNormalize       : "
    f"{VECNORM_PATH}"
)

print(
    f"Reward file        : "
    f"{REWARD_FILE}"
)

print()
print("Episode rewards:")

for i, reward in enumerate(
    callback.episode_rewards,
    start=1
):

    length = callback.episode_lengths[i - 1]

    print(
        f"  Episode {i:2d}: "
        f"{reward:,.2f} "
        f"| length = {length:,}"
    )

print("=" * 70)

SAC BASELINE — TRAINING
Training file       : /content/XRL-for-optimising-battery-schedules/data/rl/german_rl_train_2022H2_2024.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

Output()

/usr/local/lib/python3.13/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

In [8]:
import zipfile
from google.colab import files

results_dir = (
    "/content/XRL-for-optimising-battery-schedules/"
    "log/causal_rl/sac_baseline_20ep/run_01"
)
zip_path = "/content/sac_baseline_20ep_results.zip"

files_to_download = [
    "model_final.zip",
    "vecnormalize.pkl",
    "episode_rewards.csv",
]

missing = [
    name for name in files_to_download
    if not os.path.isfile(os.path.join(results_dir, name))
]

if missing:
    print("Missing files:", missing)
    print("Check that the SAC baseline training output folder exists.")
else:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for name in files_to_download:
            z.write(os.path.join(results_dir, name), arcname=name)

    print("ZIP created. Starting download...")
    files.download(zip_path)


ZIP created. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:

import os
import sys
import copy
import numpy as np
import pandas as pd
import torch

from gymnasium import spaces
from gymnasium.wrappers import FlattenObservation
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback

# ------------------------------------------------------------
# 1. Paths and training budget
# ------------------------------------------------------------
REPO = "/content/XRL-for-optimising-battery-schedules"
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_ppo

DATA_FILE = os.path.join(
    REPO, "data/rl/german_rl_combined_tft_2022_2025H1.csv"
)

OUT_DIR = os.path.join(
    REPO, "log/causal_rl/sac_tft/run_02_20ep"
)
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 22
N_EPISODES = 20

df = pd.read_csv(DATA_FILE)
N_ROWS = len(df)
TOTAL_TIMESTEPS = N_EPISODES * N_ROWS

print("=" * 64)
print("SAC + CAUSAL TFT — 20 EPISODES")
print("=" * 64)
print("Dataset rows       :", f"{N_ROWS:,}")
print("Episodes requested :", N_EPISODES)
print("Timesteps requested:", f"{TOTAL_TIMESTEPS:,}")
print("Output directory   :", OUT_DIR)

# ------------------------------------------------------------
# 2. TFT forecast columns: 24 p10 + 24 p50 + 24 p90
#    + 24 uncertainty features
# ------------------------------------------------------------
TFT_COLS = (
    [f"TFT_p10_h{i:02d}" for i in range(24)]
    + [f"TFT_p50_h{i:02d}" for i in range(24)]
    + [f"TFT_p90_h{i:02d}" for i in range(24)]
    + [f"TFT_uncertainty_h{i:02d}" for i in range(24)]
)

missing = [c for c in TFT_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing TFT columns: {missing}")

tft_values = df[TFT_COLS].to_numpy(dtype=np.float32)
if not np.isfinite(tft_values).all():
    raise ValueError("TFT features contain NaN or infinite values.")

# ------------------------------------------------------------
# 3. Environment with causal TFT forecasts
# ------------------------------------------------------------
class CausalTFTBatteryEnv(FreeBatteryEnv):
    def __init__(self, *args, **kwargs):
        kwargs["tft_24h"] = None
        super().__init__(*args, **kwargs)

        forecast_df = pd.read_csv(self.data_file)
        missing_cols = [c for c in TFT_COLS if c not in forecast_df.columns]
        if missing_cols:
            raise ValueError(f"Missing TFT columns: {missing_cols}")

        self.tft_data = forecast_df[TFT_COLS].to_numpy(
            dtype=np.float32
        )

        if len(self.tft_data) != len(self.data):
            raise ValueError(
                f"TFT rows ({len(self.tft_data)}) do not match "
                f"environment rows ({len(self.data)})."
            )

        if not np.isfinite(self.tft_data).all():
            raise ValueError("Non-finite TFT observation features.")

        # Correct Gymnasium Dict-space update.
        self.observation_space.spaces["tft_forecasts"] = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(96,),
            dtype=np.float32,
        )

    def _get_obs(self):
        obs, done = super()._get_obs()
        idx = min(self.count, len(self.tft_data) - 1)
        obs["tft_forecasts"] = self.tft_data[idx]
        return obs, done

# ------------------------------------------------------------
# 4. Environment wrappers and normalization
# ------------------------------------------------------------
env_kwargs = copy.deepcopy(de_bat_ppo)
env_kwargs["data_file"] = DATA_FILE
env_kwargs["modeling_period_h"] = N_ROWS

def make_tft_env():
    env = CausalTFTBatteryEnv(**copy.deepcopy(env_kwargs))
    env = Monitor(env)
    env = FlattenObservation(env)
    return env

vec_env = DummyVecEnv([make_tft_env])
vec_env.seed(SEED)

vec_env = VecNormalize(
    vec_env,
    norm_obs=True,
    norm_reward=True,
    gamma=0.999,
)

print("Observation shape:", vec_env.observation_space.shape)
print("Action space     :", vec_env.action_space)

# ------------------------------------------------------------
# 5. Record full-episode rewards
# ------------------------------------------------------------
class EpisodeRewardCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.rewards = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                row = {
                    "episode": len(self.rewards) + 1,
                    "reward": float(info["episode"]["r"]),
                    "length": int(info["episode"]["l"]),
                    "timesteps": self.num_timesteps,
                }
                self.rewards.append(row)
                print(
                    f"Episode {row['episode']:02d}/{N_EPISODES}: "
                    f"reward={row['reward']:,.2f}; "
                    f"length={row['length']:,}",
                    flush=True,
                )
        return True

callback = EpisodeRewardCallback()

# ------------------------------------------------------------
# 6. SAC hyperparameters — unchanged architecture
# ------------------------------------------------------------
policy_kwargs = {
    "net_arch": [256, 512, 512, 256],
    "activation_fn": torch.nn.Tanh,
    "n_critics": 2,
    "share_features_extractor": True,
}

model = SAC(
    "MlpPolicy",
    vec_env,
    learning_rate=1e-4,
    batch_size=512,
    buffer_size=500_000,
    tau=0.006985,
    target_update_interval=5,
    gamma=0.999,
    gradient_steps=-1,
    learning_starts=1000,
    train_freq=35,
    ent_coef="auto_0.1",
    target_entropy="auto",
    policy_kwargs=policy_kwargs,
    seed=SEED,
    device="cuda",
    verbose=0,
)

# ------------------------------------------------------------
# 7. Train
# ------------------------------------------------------------
print("\nStarting SAC + TFT training...", flush=True)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callback,
    progress_bar=True,
)

# ------------------------------------------------------------
# 8. Save model, normalization and episode rewards
# ------------------------------------------------------------
model_path = os.path.join(OUT_DIR, "model_final")
vecnorm_path = os.path.join(OUT_DIR, "vecnormalize.pkl")
rewards_path = os.path.join(OUT_DIR, "episode_rewards.csv")

model.save(model_path)
vec_env.save(vecnorm_path)

rewards_df = pd.DataFrame(callback.rewards)
rewards_df.to_csv(rewards_path, index=False)

print("\n" + "=" * 64)
print("SAC + TFT TRAINING COMPLETE")
print("=" * 64)
print("Episodes recorded :", f"{len(rewards_df)} / {N_EPISODES}")
print("Timesteps reached :", f"{model.num_timesteps:,}")
print("Model             :", model_path + ".zip")
print("VecNormalize      :", vecnorm_path)
print("Episode rewards   :", rewards_path)

if len(rewards_df):
    print("\nEpisode rewards:")
    print(rewards_df[["episode", "reward", "length"]].to_string(index=False))

vec_env.close()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


SAC + CAUSAL TFT — 20 EPISODES
Dataset rows       : 26,304
Episodes requested : 20
Timesteps requested: 526,080
Output directory   : /content/XRL-for-optimising-battery-schedules/log/causal_rl/sac_tft/run_02_20ep


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Observation shape: (117,)
Action space     : Box(-1.0, 1.0, (1,), float32)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Starting SAC + TFT training...


Output()

/usr/local/lib/python3.13/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

In [2]:

# ================================================================
# SAC + CAUSAL TFT — FINAL H2 2025 TEST EVALUATION
# No retraining; frozen observation normalization; deterministic policy
# ================================================================

import os
import numpy as np
import pandas as pd

from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from gymnasium.wrappers import FlattenObservation

from copy import deepcopy
from envs.env_params import de_bat_ppo

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------
REPO = "/content/XRL-for-optimising-battery-schedules"

TEST_FILE = os.path.join(
    REPO, "data/rl/german_rl_h2_2025.csv"
)

MODEL_DIR = os.path.join(
    REPO, "log/causal_rl/sac_tft/run_02_20ep"
)

MODEL_PATH = os.path.join(MODEL_DIR, "model_final.zip")
VECNORM_PATH = os.path.join(MODEL_DIR, "vecnormalize.pkl")

OUT_DIR = os.path.join(MODEL_DIR, "h2_2025_test")
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------
# 2. Validate files and test data
# ------------------------------------------------
assert os.path.isfile(TEST_FILE), f"Missing test data: {TEST_FILE}"
assert os.path.isfile(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.isfile(VECNORM_PATH), f"Missing VecNormalize: {VECNORM_PATH}"

test_df = pd.read_csv(TEST_FILE)

assert len(test_df) == 4416, (
    f"Expected 4,416 H2 test rows, found {len(test_df)}"
)

missing = [c for c in TFT_COLS if c not in test_df.columns]
assert not missing, f"Missing TFT columns in test data: {missing[:10]}"

assert np.isfinite(test_df[TFT_COLS].to_numpy(dtype=np.float32)).all(), (
    "TFT test features contain NaN or infinite values."
)

print("Test rows:", len(test_df))
print("Test period:", test_df.iloc[0].get("timestamp", "timestamp column unavailable"),
      "to", test_df.iloc[-1].get("timestamp", "timestamp column unavailable"))

# ------------------------------------------------
# 3. Build the same environment used for SAC + TFT
# ------------------------------------------------
env_kwargs = deepcopy(de_bat_ppo)
env_kwargs["data_file"] = TEST_FILE
env_kwargs["modeling_period_h"] = len(test_df)

def make_test_env():
    env = CausalTFTBatteryEnv(**env_kwargs)
    env = Monitor(env)
    env = FlattenObservation(env)
    return env

raw_env = DummyVecEnv([make_test_env])

# Load the training normalization statistics.
# Do not update them using test observations.
env = VecNormalize.load(VECNORM_PATH, raw_env)
env.training = False
env.norm_reward = False

model = SAC.load(MODEL_PATH, env=env, device="auto")

assert env.observation_space.shape == (117,), (
    f"Unexpected observation shape: {env.observation_space.shape}"
)

print("Observation shape:", env.observation_space.shape)
print("Evaluation mode: deterministic")
print("Normalization updates: disabled")

# ------------------------------------------------
# 4. Run one full test episode
# ------------------------------------------------
obs = env.reset()

rewards = []
actions = []
soc_values = []

done = np.array([False])

while not done[0]:
    action, _ = model.predict(obs, deterministic=True)

    obs, reward, done, infos = env.step(action)

    actions.append(float(np.asarray(action).reshape(-1)[0]))
    rewards.append(float(reward[0]))

    # Capture SOC if the environment exposes it in the observation.
    # The flattened observation is normalized, so SOC is not extracted here.

# VecNormalize has norm_reward=False, so rewards are unnormalized.
rewards = np.asarray(rewards, dtype=np.float64)
actions = np.asarray(actions, dtype=np.float64)

assert len(rewards) == len(test_df), (
    f"Expected 4,416 evaluation steps, got {len(rewards)}"
)

# ------------------------------------------------
# 5. Save hourly results
# ------------------------------------------------
results = test_df.iloc[:len(rewards)].copy()
results["sac_tft_action"] = actions
results["step_reward"] = rewards
results["cumulative_reward"] = np.cumsum(rewards)

hourly_path = os.path.join(OUT_DIR, "sac_tft_h2_2025_hourly.csv")
results.to_csv(hourly_path, index=False)

# ------------------------------------------------
# 6. Summary metrics
# ------------------------------------------------
summary = {
    "model": "SAC + causal TFT",
    "test_period": "2025 H2",
    "test_hours": len(rewards),
    "total_reward": float(rewards.sum()),
    "mean_reward_per_hour": float(rewards.mean()),
    "median_reward_per_hour": float(np.median(rewards)),
    "reward_std_per_hour": float(rewards.std()),
    "mean_action": float(actions.mean()),
    "action_min": float(actions.min()),
    "action_max": float(actions.max()),
    "fraction_charge": float(np.mean(actions > 0.1)),
    "fraction_idle": float(np.mean(np.abs(actions) <= 0.1)),
    "fraction_discharge": float(np.mean(actions < -0.1)),
}

summary_df = pd.DataFrame([summary])
summary_path = os.path.join(OUT_DIR, "sac_tft_h2_2025_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 64)
print("SAC + CAUSAL TFT — H2 2025 TEST RESULTS")
print("=" * 64)
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key:28s}: {value:,.6f}")
    else:
        print(f"{key:28s}: {value}")

print("\nHourly results:", hourly_path)
print("Summary:", summary_path)

env.close()

Test rows: 4416
Test period: 2025-07-01 00:00:00 to 2025-12-31 23:00:00


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Observation shape: (117,)
Evaluation mode: deterministic
Normalization updates: disabled

SAC + CAUSAL TFT — H2 2025 TEST RESULTS
model                       : SAC + causal TFT
test_period                 : 2025 H2
test_hours                  : 4416
total_reward                : 20,437.130794
mean_reward_per_hour        : 4.627973
median_reward_per_hour      : 0.000000
reward_std_per_hour         : 119.243984
mean_action                 : 0.013059
action_min                  : -0.984840
action_max                  : 0.997361
fraction_charge             : 0.501359
fraction_idle               : 0.028533
fraction_discharge          : 0.470109

Hourly results: /content/XRL-for-optimising-battery-schedules/log/causal_rl/sac_tft/run_02_20ep/h2_2025_test/sac_tft_h2_2025_hourly.csv
Summary: /content/XRL-for-optimising-battery-schedules/log/causal_rl/sac_tft/run_02_20ep/h2_2025_test/sac_tft_h2_2025_summary.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:

# ================================================================
# SAC BASELINE — H2 2025 TEST EVALUATION
# ================================================================

import os
import numpy as np
import pandas as pd
from copy import deepcopy

from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from gymnasium.wrappers import FlattenObservation

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_ppo

REPO = "/content/XRL-for-optimising-battery-schedules"

TEST_FILE = os.path.join(
    REPO, "data/rl/german_rl_h2_2025.csv"
)

MODEL_DIR = os.path.join(
    REPO, "log/causal_rl/sac_baseline_20ep/run_01"
)

MODEL_PATH = os.path.join(MODEL_DIR, "model_final.zip")
VECNORM_PATH = os.path.join(MODEL_DIR, "vecnormalize.pkl")

OUT_DIR = os.path.join(MODEL_DIR, "h2_2025_test")
os.makedirs(OUT_DIR, exist_ok=True)

# 1. Verify files and test data
for path in [TEST_FILE, MODEL_PATH, VECNORM_PATH]:
    assert os.path.isfile(path), f"Missing file: {path}"

test_df = pd.read_csv(TEST_FILE)
assert len(test_df) == 4416, f"Expected 4416 rows, got {len(test_df)}"

print("Test rows:", len(test_df))
print("Test period:", test_df.iloc[0]["timestamp"], "to",
      test_df.iloc[-1]["timestamp"])

# 2. Create baseline environment (no TFT features)
env_kwargs = deepcopy(de_bat_ppo)
env_kwargs["data_file"] = TEST_FILE
env_kwargs["modeling_period_h"] = len(test_df)

def make_test_env():
    env = FreeBatteryEnv(**env_kwargs)
    env = Monitor(env)
    env = FlattenObservation(env)
    return env

raw_env = DummyVecEnv([make_test_env])

# Load the saved training normalizer; freeze its statistics
env = VecNormalize.load(VECNORM_PATH, raw_env)
env.training = False
env.norm_reward = False

model = SAC.load(MODEL_PATH, env=env, device="auto")

assert env.observation_space.shape == (21,), (
    f"Expected baseline observation shape (21,), got {env.observation_space.shape}"
)

print("Observation shape:", env.observation_space.shape)
print("Deterministic evaluation; normalization updates disabled.")

# 3. Evaluate one full H2 episode
obs = env.reset()
rewards, actions = [], []
done = np.array([False])

while not done[0]:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, infos = env.step(action)

    actions.append(float(np.asarray(action).reshape(-1)[0]))
    rewards.append(float(reward[0]))

rewards = np.asarray(rewards, dtype=np.float64)
actions = np.asarray(actions, dtype=np.float64)

assert len(rewards) == 4416, f"Expected 4416 steps, got {len(rewards)}"

# 4. Save hourly results
results = test_df.iloc[:len(rewards)].copy()
results["sac_baseline_action"] = actions
results["step_reward"] = rewards
results["cumulative_reward"] = np.cumsum(rewards)

hourly_path = os.path.join(OUT_DIR, "sac_baseline_h2_2025_hourly.csv")
results.to_csv(hourly_path, index=False)

# 5. Calculate summary metrics
summary = {
    "model": "SAC baseline",
    "test_period": "2025 H2",
    "test_hours": len(rewards),
    "total_reward": float(rewards.sum()),
    "mean_reward_per_hour": float(rewards.mean()),
    "median_reward_per_hour": float(np.median(rewards)),
    "reward_std_per_hour": float(rewards.std()),
    "mean_action": float(actions.mean()),
    "action_min": float(actions.min()),
    "action_max": float(actions.max()),
    "fraction_charge": float(np.mean(actions > 0.1)),
    "fraction_idle": float(np.mean(np.abs(actions) <= 0.1)),
    "fraction_discharge": float(np.mean(actions < -0.1)),
}

summary_path = os.path.join(OUT_DIR, "sac_baseline_h2_2025_summary.csv")
pd.DataFrame([summary]).to_csv(summary_path, index=False)

print("\n" + "=" * 60)
print("SAC BASELINE — H2 2025 TEST RESULTS")
print("=" * 60)
for key, value in summary.items():
    print(f"{key:28s}: {value:,.6f}" if isinstance(value, float)
          else f"{key:28s}: {value}")

print("\nHourly results:", hourly_path)
print("Summary:", summary_path)

env.close()


Test rows: 4416
Test period: 2025-07-01 00:00:00 to 2025-12-31 23:00:00
Observation shape: (21,)
Deterministic evaluation; normalization updates disabled.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



SAC BASELINE — H2 2025 TEST RESULTS
model                       : SAC baseline
test_period                 : 2025 H2
test_hours                  : 4416
total_reward                : 17,522.245722
mean_reward_per_hour        : 3.967900
median_reward_per_hour      : 0.000000
reward_std_per_hour         : 97.197977
mean_action                 : -0.188703
action_min                  : -0.993165
action_max                  : 0.997221
fraction_charge             : 0.306386
fraction_idle               : 0.024230
fraction_discharge          : 0.669384

Hourly results: /content/XRL-for-optimising-battery-schedules/log/causal_rl/sac_baseline_20ep/run_01/h2_2025_test/sac_baseline_h2_2025_hourly.csv
Summary: /content/XRL-for-optimising-battery-schedules/log/causal_rl/sac_baseline_20ep/run_01/h2_2025_test/sac_baseline_h2_2025_summary.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
import zipfile
from google.colab import files

results_dir = (
    "/content/XRL-for-optimising-battery-schedules/"
    "log/causal_rl/sac_tft/run_02_20ep"
)
zip_path = "/content/sac_baseline_20ep_results.zip"

files_to_download = [
    "model_final.zip",
    "vecnormalize.pkl",
    "episode_rewards.csv",
]

missing = [
    name for name in files_to_download
    if not os.path.isfile(os.path.join(results_dir, name))
]

if missing:
    print("Missing files:", missing)
    print("Check that the SAC baseline training output folder exists.")
else:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for name in files_to_download:
            z.write(os.path.join(results_dir, name), arcname=name)

    print("ZIP created. Starting download...")
    files.download(zip_path)

ZIP created. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:

# ================================================================
# CEM BASELINE — TRAIN ON 2022–2024 ONLY
# Continuous actions, 21-feature state, no TFT
# ================================================================

import os
import numpy as np
import pandas as pd
from copy import deepcopy

from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_ppo

# ----------------------------
# Configuration
# ----------------------------
REPO = "/content/XRL-for-optimising-battery-schedules"

TRAIN_FILE = os.path.join(
    REPO, "data/rl/german_rl_train_2022H2_2024.csv"
)

OUT_DIR = os.path.join(
    REPO, "log/causal_rl/cem_baseline/run_01"
)
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 22
POPULATION_SIZE = 32
ELITE_COUNT = 6
N_ITERATIONS = 20
SMOOTHING = 0.7
INITIAL_STD = 0.5

rng = np.random.default_rng(SEED)

# ----------------------------
# Check training data
# ----------------------------
assert os.path.isfile(TRAIN_FILE), f"Missing training file: {TRAIN_FILE}"

train_df = pd.read_csv(TRAIN_FILE)
N_ROWS = len(train_df)

assert N_ROWS == 21960, f"Expected 21,960 rows, got {N_ROWS}"

env_kwargs = deepcopy(de_bat_ppo)
env_kwargs["data_file"] = TRAIN_FILE
env_kwargs["modeling_period_h"] = N_ROWS

def make_env():
    return FreeBatteryEnv(**env_kwargs)

def flatten_obs(obs):
    return np.concatenate([
        np.asarray(obs[key], dtype=np.float64).reshape(-1)
        for key in obs
    ])

env = make_env()

base_env = env.venv.envs[0] if hasattr(env, "venv") else env

print("Observation features:", len(base_env.observation_space.spaces))
print("Action space:", base_env.action_space)

assert env.action_space.shape == (1,), (
    f"Expected one continuous action, got {env.action_space}"
)

print("CEM baseline training")
print("Training rows:", N_ROWS)
print("Observation features:", len(base_env.observation_space.spaces))
print("Population:", POPULATION_SIZE)
print("Elite policies:", ELITE_COUNT)
print("Iterations:", N_ITERATIONS)

# ----------------------------
# Fixed observation scaling
# Avoid raw feature magnitudes dominating
# the linear policy.
# ----------------------------
obs_samples = []

obs, _ = env.reset(seed=SEED)
obs_samples.append(flatten_obs(obs))

for _ in range(min(2000, N_ROWS - 1)):
    random_action = env.action_space.sample()
    obs, _, terminated, truncated, _ = env.step(random_action)
    obs_samples.append(flatten_obs(obs))
    if terminated or truncated:
        break

obs_samples = np.asarray(obs_samples)
obs_mean = obs_samples.mean(axis=0)
obs_std = obs_samples.std(axis=0)
obs_std[obs_std < 1e-6] = 1.0

env.close()

# ----------------------------
# Policy parameters:
# 21 normalized features + bias
# ----------------------------
N_PARAMS = 22

def evaluate_policy(theta, seed):
    evaluation_env = make_env()
    obs, _ = evaluation_env.reset(seed=seed)
    total_reward = 0.0
    steps = 0

    while True:
        obs = flatten_obs(obs)
        normalized_obs = (obs - obs_mean) / obs_std
        features = np.append(normalized_obs, 1.0)

        action = np.array(
            [np.tanh(np.dot(theta, features))],
            dtype=np.float32
        )

        obs, reward, terminated, truncated, _ = evaluation_env.step(action)
        total_reward += float(reward)
        steps += 1

        if terminated or truncated:
            break

    evaluation_env.close()
    return total_reward, steps

# ----------------------------
# Cross-Entropy Method
# ----------------------------
mean = np.zeros(N_PARAMS, dtype=np.float64)
std = np.full(N_PARAMS, INITIAL_STD, dtype=np.float64)

history = []
best_reward = -np.inf
best_theta = None

for iteration in range(1, N_ITERATIONS + 1):
    candidates = rng.normal(
        loc=mean,
        scale=std,
        size=(POPULATION_SIZE, N_PARAMS)
    )

    scores = []
    for i, theta in enumerate(candidates, start=1):
        reward, steps = evaluate_policy(theta, seed=SEED)
        scores.append(reward)

    scores = np.asarray(scores)
    elite_indices = np.argsort(scores)[-ELITE_COUNT:]
    elites = candidates[elite_indices]

    # Update sampling distribution toward elite candidates
    new_mean = elites.mean(axis=0)
    new_std = elites.std(axis=0) + 1e-3

    mean = SMOOTHING * mean + (1.0 - SMOOTHING) * new_mean
    std = SMOOTHING * std + (1.0 - SMOOTHING) * new_std

    iteration_best_idx = int(np.argmax(scores))
    iteration_best_reward = float(scores[iteration_best_idx])

    if iteration_best_reward > best_reward:
        best_reward = iteration_best_reward
        best_theta = candidates[iteration_best_idx].copy()

    history.append({
        "iteration": iteration,
        "best_reward": iteration_best_reward,
        "mean_population_reward": float(scores.mean()),
        "elite_mean_reward": float(scores[elite_indices].mean()),
        "population_std_reward": float(scores.std()),
        "best_reward_so_far": best_reward,
    })

    print(
        f"Iteration {iteration:02d}/{N_ITERATIONS} | "
        f"best={iteration_best_reward:,.2f} | "
        f"population mean={scores.mean():,.2f} | "
        f"best so far={best_reward:,.2f}"
    )

# ----------------------------
# Save policy and training log
# ----------------------------
policy_path = os.path.join(OUT_DIR, "cem_policy.npz")
np.savez(
    policy_path,
    theta=best_theta,
    obs_mean=obs_mean,
    obs_std=obs_std,
    seed=SEED,
    population_size=POPULATION_SIZE,
    elite_count=ELITE_COUNT,
    iterations=N_ITERATIONS,
)

history_path = os.path.join(OUT_DIR, "cem_training_history.csv")
pd.DataFrame(history).to_csv(history_path, index=False)

print("\n" + "=" * 60)
print("CEM BASELINE TRAINING COMPLETE")
print("=" * 60)
print("Best training episode reward:", f"{best_reward:,.2f}")
print("Policy saved:", policy_path)
print("Training history:", history_path)
print("\nNext: evaluate the saved policy on 2025 H2.")


Observation features: 21
Action space: Box(-1.0, 1.0, (1,), float32)
CEM baseline training
Training rows: 21960
Observation features: 21
Population: 32
Elite policies: 6
Iterations: 20


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Iteration 01/20 | best=-10,344.79 | population mean=-281,734.83 | best so far=-10,344.79


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Iteration 02/20 | best=11,456.76 | population mean=-193,555.87 | best so far=11,456.76


...

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Iteration 19/20 | best=38,884.15 | population mean=28,274.61 | best so far=39,882.47


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Iteration 20/20 | best=40,947.71 | population mean=26,723.16 | best so far=40,947.71

CEM BASELINE TRAINING COMPLETE
Best training episode reward: 40,947.71
Policy saved: /content/XRL-for-optimising-battery-schedules/log/causal_rl/cem_baseline/run_01/cem_policy.npz
Training history: /content/XRL-for-optimising-battery-schedules/log/causal_rl/cem_baseline/run_01/cem_training_history.csv

Next: evaluate the saved policy on 2025 H2.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [15]:

# ================================================================
# CEM BASELINE — FINAL EVALUATION ON 2025 H2
# ================================================================

import os
import numpy as np
import pandas as pd
from copy import deepcopy
from envs.environments import FreeBatteryEnv
from envs.env_params import de_bat_ppo

REPO = "/content/XRL-for-optimising-battery-schedules"
POLICY_DIR = os.path.join(REPO, "log/causal_rl/cem_baseline/run_01")
POLICY_PATH = os.path.join(POLICY_DIR, "cem_policy.npz")
TEST_FILE = os.path.join(REPO, "data/rl/german_rl_h2_2025.csv")
OUT_DIR = os.path.join(POLICY_DIR, "h2_2025_test")
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.isfile(POLICY_PATH), f"Missing policy: {POLICY_PATH}"
assert os.path.isfile(TEST_FILE), f"Missing test file: {TEST_FILE}"

policy = np.load(POLICY_PATH)
theta = policy["theta"]
obs_mean = policy["obs_mean"]
obs_std = policy["obs_std"]

test_df = pd.read_csv(TEST_FILE)
assert len(test_df) == 4416, f"Expected 4416 H2 rows, got {len(test_df)}"

def flatten_obs(obs):
    return np.concatenate([
        np.asarray(obs[key], dtype=np.float64).reshape(-1)
        for key in obs
    ])

env_kwargs = deepcopy(de_bat_ppo)
env_kwargs["data_file"] = TEST_FILE
env_kwargs["modeling_period_h"] = len(test_df)
env = FreeBatteryEnv(**env_kwargs)

obs, _ = env.reset(seed=22)
rows = []
total_reward = 0.0
step = 0

while True:
    flat_obs = flatten_obs(obs)
    normalized_obs = (flat_obs - obs_mean) / obs_std
    features = np.append(normalized_obs, 1.0)
    action = np.array(
        [np.tanh(np.dot(theta, features))],
        dtype=np.float32
    )

    obs, reward, terminated, truncated, info = env.step(action)
    reward_value = float(np.asarray(reward).reshape(-1)[0])
    total_reward += reward_value

    rows.append({
        "step": step,
        "action": float(action[0]),
        "reward": reward_value,
    })
    step += 1

    if terminated or truncated:
        break

env.close()

hourly = pd.DataFrame(rows)

# Attach test timestamps when available
if "timestamp" in test_df.columns:
    hourly.insert(1, "timestamp", test_df["timestamp"].iloc[:len(hourly)].to_numpy())
elif "time_idx" in test_df.columns:
    hourly.insert(1, "time_idx", test_df["time_idx"].iloc[:len(hourly)].to_numpy())

hourly_path = os.path.join(OUT_DIR, "cem_h2_2025_hourly.csv")
hourly.to_csv(hourly_path, index=False)

summary = pd.DataFrame([{
    "model": "CEM baseline",
    "test_rows": len(hourly),
    "total_reward": total_reward,
    "mean_reward_per_hour": hourly["reward"].mean(),
    "std_reward_per_hour": hourly["reward"].std(),
    "mean_action": hourly["action"].mean(),
    "charge_fraction": (hourly["action"] > 0.1).mean(),
    "idle_fraction": (hourly["action"].abs() <= 0.1).mean(),
    "discharge_fraction": (hourly["action"] < -0.1).mean(),
}])

summary_path = os.path.join(OUT_DIR, "cem_h2_2025_summary.csv")
summary.to_csv(summary_path, index=False)

print("CEM H2 2025 evaluation complete")
print(summary.to_string(index=False))
print("\nHourly results:", hourly_path)
print("Summary:", summary_path)

CEM H2 2025 evaluation complete
       model  test_rows  total_reward  mean_reward_per_hour  std_reward_per_hour  mean_action  charge_fraction  idle_fraction  discharge_fraction
CEM baseline       4416   -642.707746             -0.145541             9.671615    -0.999232              0.0            0.0                 1.0

Hourly results: /content/XRL-for-optimising-battery-schedules/log/causal_rl/cem_baseline/run_01/h2_2025_test/cem_h2_2025_hourly.csv
Summary: /content/XRL-for-optimising-battery-schedules/log/causal_rl/cem_baseline/run_01/h2_2025_test/cem_h2_2025_summary.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
